# Mortgage Approval: End-to-End MLOps on Snowflake

This notebook walks one model through its full lifecycle on Snowflake, using the mortgage-approval assets in `ML_DEMO.PUBLIC`:

```
MORTGAGE_LENDING_DEMO_DATA (table)  ◄── stored procedures: API ingest + clean/MERGE (task)
   └─► MORTGAGE_FEATURE_VIEW v5   (Feature Store: Dynamic Table + data-quality checks)
          └─► MORTGAGE_APPROVAL_TRAINING_DS   (versioned, immutable Dataset)
                 └─► Experiment runs ─► MORTGAGE_APPROVAL_CLASSIFIER   (Model Registry; aliases PRODUCTION / CHALLENGER)
                                           ├─► SQL / Python batch inference (warehouse)
                                           ├─► MORTGAGE_APPROVAL_SERVICE   (REST, multi-instance SPCS)
                                           └─► Model Monitor ─► Alert ─► Review & retraining workflow ─► alias promotion
```

| # | Section | Requirement covered |
|---|---|---|
| 1 | Data ingestion & EDA | Explore with SQL and pandas, both visible in query history; **1d:** bring data in and prepare it with Python and SQL stored procedures |
| 2 | Feature engineering | The SQL behind the features in the Feature Store |
| 3 | Feature data quality | DQ monitoring on the feature store |
| 4 | Existing model compatibility | Import a model trained elsewhere (SageMaker example + a parity test that runs) |
| 5 | Model training | Dataset reproducibility, experiment tracking and reruns, lineage in code and UI |
| 6 | Model inference | Command-line calls and latency, scaling across instances, failure modes |
| 7 | Monitoring | Operational, model and **custom** metrics, simulated drift, owner alert, retraining and review workflow |
| 8 | Promotion with aliases | Promote and roll back via `PRODUCTION` / `CHALLENGER` aliases in SQL, Python, services and gateways |
| 9 | Governance | One RBAC model across data, features, models, services and monitors; USAGE vs READ vs OWNERSHIP on models |
| 10 | Summary | Requirement recap and cleanup |

**💡 Where Snowflake differentiates** callouts point out the steps that need extra systems or hand-built plumbing on other platforms.

**Runtime:** Snowflake Workspace notebook on Container Runtime (CPU). Run top to bottom; reruns are safe because each step reuses objects that already exist. Cells that would build images or call external systems are guarded by a `RUN_…`/`DEPLOY_…` flag that defaults to `False`.

## 0. Setup

One session and one set of names for the whole notebook. Every query runs under the query tag `mortgage_mlops_demo`, so the full audit trail can be pulled from query history later.

In [ ]:
import json, sys, time, hashlib, inspect, platform
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()

DB, SCHEMA, WH, ROLE = "ML_DEMO", "PUBLIC", "COMPUTE_WH", "SYSADMIN"
QUERY_TAG = "mortgage_mlops_demo"

SOURCE_TABLE        = f"{DB}.{SCHEMA}.MORTGAGE_LENDING_DEMO_DATA"
FV_NAME, FV_VERSION = "MORTGAGE_FEATURE_VIEW", "5"
FV_TABLE            = f'{DB}.{SCHEMA}."{FV_NAME}${FV_VERSION}"'
DATASET_NAME        = "MORTGAGE_APPROVAL_TRAINING_DS"
DATASET_VERSION     = "V2"
MODEL_NAME          = "MORTGAGE_APPROVAL_CLASSIFIER"
TRAIN_VERSION       = "V3"          # version trained and registered in Section 5
SERVING_VERSION     = "V2"          # version behind the live REST service (Section 6)
SERVICE_NAME        = "MORTGAGE_APPROVAL_SERVICE"
COMPUTE_POOL        = "MORTGAGE_INFERENCE"
EXTERNAL_MODEL_NAME = "MORTGAGE_APPROVAL_EXTERNAL_XGB"
EXPERIMENT_NAME     = "MORTGAGE_APPROVAL_EXPERIMENT"
LABEL = "MORTGAGERESPONSE"
FEATURES = ["MONTH", "DAY_OF_YEAR", "DOTW", "LOAN_AMOUNT", "INCOME", "INCOME_LOAN_RATIO",
            "MEAN_COUNTY_INCOME", "HIGH_INCOME_FLAG", "AVG_THIRTY_DAY_LOAN_AMOUNT"]
OWNER_EMAIL = "tom.smith@snowflake.com"   # model owner who receives alerts in Section 7

session.use_role(ROLE)
session.use_warehouse(WH)
session.use_database(DB)
session.use_schema(SCHEMA)
session.sql(f"ALTER SESSION SET QUERY_TAG = '{QUERY_TAG}'").collect()

def q(sql: str) -> pd.DataFrame:
    """Run SQL and return pandas (small results only)."""
    return session.sql(sql).to_pandas()

def show(sql: str) -> pd.DataFrame:
    """Run a SHOW/DESCRIBE command and return pandas."""
    return pd.DataFrame([r.as_dict() for r in session.sql(sql).collect()])

print(q("SELECT CURRENT_ACCOUNT() AS account, CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse"))
print("Python", platform.python_version(), "| snowflake-ml-python", version("snowflake-ml-python"),
      "| xgboost", version("xgboost"), "| scikit-learn", version("scikit-learn"))

## 1. Data ingestion & exploratory analysis

The raw HMDA-style loan applications are already in Snowflake (`MORTGAGE_LENDING_DEMO_DATA`). We explore them two ways:

* **SQL**: profiling and aggregations run in the warehouse.
* **pandas / Snowpark DataFrames**: the familiar Python API. Snowpark pushes the work down as SQL, and only the sample we chart is pulled into memory.

Both paths leave the same audit trail in **query history**, which we show at the end of this section.

In [ ]:
# 1a. SQL profiling of the raw table
profile = q(f"""
SELECT COUNT(*)                                   AS loan_applications,
       COUNT(DISTINCT COUNTY_NAME)                AS counties,
       MIN(TO_TIMESTAMP(TS))::DATE                AS first_application,
       MAX(TO_TIMESTAMP(TS))::DATE                AS last_application,
       ROUND(AVG(MORTGAGERESPONSE), 4)            AS approval_rate,
       ROUND(COUNT_IF(APPLICANT_INCOME_000S IS NULL) / COUNT(*), 4) AS income_null_rate,
       MEDIAN(LOAN_AMOUNT_000S) * 1000            AS median_loan_amount,
       MEDIAN(APPLICANT_INCOME_000S) * 1000       AS median_income
FROM {SOURCE_TABLE}""")
profile

In [ ]:
# 1b. Approval rate by loan type and purpose (SQL aggregation)
by_segment = q(f"""
SELECT LOAN_TYPE_NAME, LOAN_PURPOSE_NAME,
       COUNT(*) AS applications,
       ROUND(AVG(MORTGAGERESPONSE), 3) AS approval_rate,
       ROUND(MEDIAN(LOAN_AMOUNT_000S) * 1000) AS median_loan
FROM {SOURCE_TABLE}
GROUP BY 1, 2
ORDER BY applications DESC""")
by_segment

In [ ]:
# 1c. pandas-style exploration: Snowpark DataFrame (pushed down) -> pandas sample for plotting
raw = session.table(SOURCE_TABLE)
print("Columns:", raw.columns)

monthly = (raw.with_column("MONTH", F.date_trunc("month", F.to_timestamp("TS")))
              .group_by("MONTH")
              .agg(F.count("*").alias("APPLICATIONS"), F.avg(LABEL).alias("APPROVAL_RATE"))
              .sort("MONTH")
              .to_pandas())

# Only a 50k-row sample comes into memory for the distribution plots
sample_pdf = raw.sample(n=50000).to_pandas()
sample_pdf["LOAN_AMOUNT"] = sample_pdf["LOAN_AMOUNT_000S"] * 1000
sample_pdf["INCOME"] = sample_pdf["APPLICANT_INCOME_000S"] * 1000

monthly

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for label, grp in sample_pdf.groupby(LABEL):
    axes[0].hist(np.log10(grp["LOAN_AMOUNT"].clip(lower=1000)), bins=50, alpha=0.6,
                 label="approved" if label == 1 else "denied", density=True)
axes[0].set_title("log10(loan amount) by outcome"); axes[0].legend()
for label, grp in sample_pdf.dropna(subset=["INCOME"]).groupby(LABEL):
    axes[1].hist(np.log10(grp["INCOME"].clip(lower=1000)), bins=50, alpha=0.6,
                 label="approved" if label == 1 else "denied", density=True)
axes[1].set_title("log10(income) by outcome"); axes[1].legend()
axes[2].bar(monthly["MONTH"].dt.strftime("%Y-%m"), monthly["APPLICATIONS"], color="#29B5E8")
ax2 = axes[2].twinx(); ax2.plot(monthly["MONTH"].dt.strftime("%Y-%m"), monthly["APPROVAL_RATE"], color="#11567F", marker="o")
axes[2].set_title("Monthly volume (bars) and approval rate (line)"); axes[2].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

### Every exploration step is in query history

The SQL above *and* the pandas/Snowpark calls ran as SQL in the warehouse, all under this session's query tag. The same trail is in Snowsight under **Monitoring » Query History** (filter by query tag `mortgage_mlops_demo`), and in `ACCOUNT_USAGE.QUERY_HISTORY` / `ACCESS_HISTORY` for audit.

In [ ]:
history = q(f"""
SELECT start_time, query_type, ROUND(total_elapsed_time/1000, 2) AS seconds, rows_produced,
       REGEXP_REPLACE(LEFT(query_text, 140), $$\s+$$, ' ') AS query_text
FROM TABLE({DB}.INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 500))
WHERE query_tag = '{QUERY_TAG}' AND query_type = 'SELECT' AND execution_status = 'SUCCESS'
ORDER BY start_time""")
history.tail(12)

> **💡 Where Snowflake differentiates**
>
> * Exploration runs where the data lives. There's no export to a separate notebook cluster, and no second copy of the data to secure.
> * SQL users and Python users hit the **same governed table** under the same RBAC, masking, and row-access policies. Other platforms often split these personas across different engines and permission models.
> * Every pandas-style operation is pushed down and **audited in one query history**, which helps in regulated lending reviews.

### 1d. Bring data in and prepare it with stored procedures

In production, data rarely arrives in a table that's ready for modeling. Two stored procedures keep that work inside Snowflake:

| Step | Procedure | Language | Why a stored procedure |
|---|---|---|---|
| **Ingest** | `INGEST_MORTGAGE_API(start_date)` | Python | Calls an external REST API with `requests`. Egress is limited to one host by an External Access Integration, and the API key comes from a Snowflake **secret**, never from code. |
| **Prepare** | `PREP_MORTGAGE_APPLICATIONS(source, target)` | SQL | Set-based cleaning that runs entirely in the warehouse: type casting, de-duplication, null-income imputation, derived ratios, then an incremental, idempotent `MERGE`. |

A **task** chains them on a schedule, so the pipeline needs no external scheduler or ETL cluster. The API call below is illustrative (a fake endpoint). The preparation procedure runs for real against `MORTGAGE_LENDING_DEMO_DATA`.

In [ ]:
RUN_API_INGEST_EXAMPLE = False   # illustrative only: api.example-lender-data.com is a placeholder host

# --- 1. Governed egress: allow exactly one host, keep the API key in a secret ---
EGRESS_DDL = f"""
CREATE OR REPLACE NETWORK RULE {DB}.{SCHEMA}.LENDER_API_RULE
  MODE = EGRESS TYPE = HOST_PORT VALUE_LIST = ('api.example-lender-data.com:443');

CREATE OR REPLACE SECRET {DB}.{SCHEMA}.LENDER_API_KEY
  TYPE = GENERIC_STRING SECRET_STRING = '<api-key-from-your-vault>';

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION LENDER_API_EAI
  ALLOWED_NETWORK_RULES = ({DB}.{SCHEMA}.LENDER_API_RULE)
  ALLOWED_AUTHENTICATION_SECRETS = ({DB}.{SCHEMA}.LENDER_API_KEY)
  ENABLED = TRUE;
"""

# --- 2. Python procedure: page through the API and land raw rows (same shape as MORTGAGE_LENDING_DEMO_DATA) ---
INGEST_PROC_DDL = f"""
CREATE OR REPLACE PROCEDURE {DB}.{SCHEMA}.INGEST_MORTGAGE_API(START_DATE DATE)
RETURNS OBJECT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests', 'pandas')
HANDLER = 'run'
EXTERNAL_ACCESS_INTEGRATIONS = (LENDER_API_EAI)
SECRETS = ('api_key' = {DB}.{SCHEMA}.LENDER_API_KEY)
AS
$$
import _snowflake
import pandas as pd
import requests

BASE_URL = "https://api.example-lender-data.com/v1/applications"

def run(session, start_date):
    headers = {{"Authorization": f"Bearer {{_snowflake.get_generic_secret_string('api_key')}}"}}
    rows, page = [], 1
    while True:
        resp = requests.get(BASE_URL, headers=headers, timeout=30,
                            params={{"updated_since": str(start_date), "page": page, "page_size": 1000}})
        resp.raise_for_status()
        body = resp.json()
        rows.extend({{
            "LOAN_ID": r["loan_id"], "TS": r["submitted_at"],
            "LOAN_TYPE_NAME": r.get("loan_type"), "LOAN_PURPOSE_NAME": r.get("loan_purpose"),
            "APPLICANT_INCOME_000S": r.get("income_thousands"), "LOAN_AMOUNT_000S": r.get("amount_thousands"),
            "COUNTY_NAME": r.get("county"), "MORTGAGERESPONSE": r.get("approved"),
        }} for r in body["data"])
        if not body.get("next_page"):
            break
        page += 1

    if not rows:
        return {{"rows_landed": 0}}
    # Overwrite the landing table each run; PREP_MORTGAGE_APPLICATIONS merges it incrementally
    session.write_pandas(pd.DataFrame(rows), "MORTGAGE_APPLICATIONS_LANDING",
                         auto_create_table=True, overwrite=True)
    return {{"rows_landed": len(rows), "pages": page}}
$$;
"""

if RUN_API_INGEST_EXAMPLE:
    for stmt in [s for s in EGRESS_DDL.split(";") if s.strip()]:
        session.sql(stmt).collect()
    session.sql(INGEST_PROC_DDL).collect()
    print(session.call(f"{DB}.{SCHEMA}.INGEST_MORTGAGE_API", "2026-10-01"))
else:
    print("Illustrative: set RUN_API_INGEST_EXAMPLE = True with a real host and key to create and call it.")
    print(INGEST_PROC_DDL[:700], "…")

In [ ]:
CLEAN_TABLE = f"{DB}.{SCHEMA}.MORTGAGE_APPLICATIONS_CLEAN"

session.sql(f"""
CREATE OR REPLACE PROCEDURE {DB}.{SCHEMA}.PREP_MORTGAGE_APPLICATIONS(SOURCE_TABLE VARCHAR, TARGET_TABLE VARCHAR)
RETURNS OBJECT
LANGUAGE SQL
COMMENT = 'Cleans raw loan applications (types, dedupe, null-income imputation) and MERGEs them into a modeling-ready table'
AS
$$
DECLARE
  inserted INTEGER DEFAULT 0;
  updated  INTEGER DEFAULT 0;
BEGIN
  CREATE TABLE IF NOT EXISTS IDENTIFIER(:TARGET_TABLE) (
    LOAN_ID NUMBER PRIMARY KEY, APPLICATION_TS TIMESTAMP_NTZ, LOAN_TYPE VARCHAR, LOAN_PURPOSE VARCHAR,
    COUNTY_NAME VARCHAR, LOAN_AMOUNT_USD NUMBER(12,0), INCOME_USD NUMBER(12,0), INCOME_IMPUTED BOOLEAN,
    INCOME_LOAN_RATIO FLOAT, MORTGAGE_APPROVED NUMBER(1,0), LOADED_AT TIMESTAMP_NTZ);

  MERGE INTO IDENTIFIER(:TARGET_TABLE) t
  USING (
    WITH typed AS (
      SELECT LOAN_ID,
             TRY_TO_TIMESTAMP_NTZ(TS)               AS APPLICATION_TS,
             INITCAP(TRIM(LOAN_TYPE_NAME))          AS LOAN_TYPE,
             INITCAP(TRIM(LOAN_PURPOSE_NAME))       AS LOAN_PURPOSE,
             UPPER(TRIM(COUNTY_NAME))               AS COUNTY_NAME,
             LOAN_AMOUNT_000S * 1000                AS LOAN_AMOUNT_USD,
             APPLICANT_INCOME_000S * 1000           AS INCOME_RAW,
             MORTGAGERESPONSE                       AS MORTGAGE_APPROVED
      FROM IDENTIFIER(:SOURCE_TABLE)
      WHERE LOAN_ID IS NOT NULL AND LOAN_AMOUNT_000S > 0 AND TRY_TO_TIMESTAMP_NTZ(TS) IS NOT NULL
      QUALIFY ROW_NUMBER() OVER (PARTITION BY LOAN_ID ORDER BY TS DESC) = 1   -- keep latest record per loan
    )
    SELECT LOAN_ID, APPLICATION_TS, LOAN_TYPE, LOAN_PURPOSE, COUNTY_NAME, LOAN_AMOUNT_USD,
           COALESCE(INCOME_RAW,
                    MEDIAN(INCOME_RAW) OVER (PARTITION BY COUNTY_NAME),
                    MEDIAN(INCOME_RAW) OVER ())                        AS INCOME_USD,
           INCOME_RAW IS NULL                                          AS INCOME_IMPUTED,
           DIV0NULL(INCOME_USD, LOAN_AMOUNT_USD)                       AS INCOME_LOAN_RATIO,
           MORTGAGE_APPROVED
    FROM typed
  ) s
  ON t.LOAN_ID = s.LOAN_ID
  WHEN MATCHED AND (t.APPLICATION_TS IS DISTINCT FROM s.APPLICATION_TS
                    OR t.INCOME_USD IS DISTINCT FROM s.INCOME_USD
                    OR t.MORTGAGE_APPROVED IS DISTINCT FROM s.MORTGAGE_APPROVED) THEN UPDATE SET
    APPLICATION_TS = s.APPLICATION_TS, LOAN_TYPE = s.LOAN_TYPE, LOAN_PURPOSE = s.LOAN_PURPOSE,
    COUNTY_NAME = s.COUNTY_NAME, LOAN_AMOUNT_USD = s.LOAN_AMOUNT_USD, INCOME_USD = s.INCOME_USD,
    INCOME_IMPUTED = s.INCOME_IMPUTED, INCOME_LOAN_RATIO = s.INCOME_LOAN_RATIO,
    MORTGAGE_APPROVED = s.MORTGAGE_APPROVED, LOADED_AT = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
  WHEN NOT MATCHED THEN INSERT VALUES (
    s.LOAN_ID, s.APPLICATION_TS, s.LOAN_TYPE, s.LOAN_PURPOSE, s.COUNTY_NAME, s.LOAN_AMOUNT_USD, s.INCOME_USD,
    s.INCOME_IMPUTED, s.INCOME_LOAN_RATIO, s.MORTGAGE_APPROVED, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ);

  SELECT "number of rows inserted", "number of rows updated" INTO :inserted, :updated
  FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

  RETURN OBJECT_CONSTRUCT('source', :SOURCE_TABLE, 'target', :TARGET_TABLE,
                          'rows_inserted', :inserted, 'rows_updated', :updated);
END;
$$""").collect()

# Demo reset so the first call shows a full load; in production the target is never dropped
session.sql(f"DROP TABLE IF EXISTS {CLEAN_TABLE}").collect()

# Idempotent: the first call loads everything; the rerun finds nothing changed and touches 0 rows
print("run 1 (initial load):", json.loads(session.call(f"{DB}.{SCHEMA}.PREP_MORTGAGE_APPLICATIONS", SOURCE_TABLE, CLEAN_TABLE)))
print("run 2 (rerun)       :", json.loads(session.call(f"{DB}.{SCHEMA}.PREP_MORTGAGE_APPLICATIONS", SOURCE_TABLE, CLEAN_TABLE)))

q(f"""
SELECT COUNT(*) AS rows_clean, COUNT_IF(INCOME_IMPUTED) AS income_imputed, COUNT_IF(INCOME_USD IS NULL) AS income_still_null,
       MIN(APPLICATION_TS)::DATE AS first_app, MAX(APPLICATION_TS)::DATE AS last_app,
       ROUND(MEDIAN(INCOME_LOAN_RATIO), 3) AS median_income_loan_ratio, ROUND(AVG(MORTGAGE_APPROVED), 4) AS approval_rate
FROM {CLEAN_TABLE}""").T.rename(columns={0: "value"})

In [ ]:
# Root task runs the preparation step nightly. With the API example enabled, ingest becomes the root and prep runs AFTER it.
session.sql(f"""
CREATE OR REPLACE TASK {DB}.{SCHEMA}.MORTGAGE_PREP_TASK
  WAREHOUSE = {WH}
  SCHEDULE = 'USING CRON 0 2 * * * America/Los_Angeles'
  COMMENT = 'Nightly: clean + MERGE raw loan applications into MORTGAGE_APPLICATIONS_CLEAN'
AS
  CALL {DB}.{SCHEMA}.PREP_MORTGAGE_APPLICATIONS('{SOURCE_TABLE}', '{CLEAN_TABLE}')""").collect()

if RUN_API_INGEST_EXAMPLE:   # ingest -> prep task graph
    session.sql(f"""CREATE OR REPLACE TASK {DB}.{SCHEMA}.MORTGAGE_INGEST_TASK WAREHOUSE = {WH}
                    SCHEDULE = 'USING CRON 0 2 * * * America/Los_Angeles'
                    AS CALL {DB}.{SCHEMA}.INGEST_MORTGAGE_API(DATEADD('day', -1, CURRENT_DATE()))""").collect()
    session.sql(f"ALTER TASK {DB}.{SCHEMA}.MORTGAGE_PREP_TASK UNSET SCHEDULE").collect()
    session.sql(f"ALTER TASK {DB}.{SCHEMA}.MORTGAGE_PREP_TASK ADD AFTER {DB}.{SCHEMA}.MORTGAGE_INGEST_TASK").collect()

# Tasks are created SUSPENDED; `ALTER TASK ... RESUME` would start the schedule
show(f"SHOW TASKS LIKE 'MORTGAGE_%' IN SCHEMA {DB}.{SCHEMA}")[["name", "state", "schedule", "predecessors", "definition"]]

> **💡 Where Snowflake differentiates**
>
> * **Ingestion and preparation run where the data lands.** A Python procedure calls the API and a SQL procedure cleans the result, both on warehouse compute. There's no separate ETL cluster or ingestion VM, and no data leaves the platform to be cleaned.
> * **Egress and credentials are governed objects.** A network rule allows one host, and the API key lives in a secret that only the procedure can read. Both are controlled by RBAC and audited like any other object.
> * **Python and SQL are interchangeable building blocks.** Either language gives the same procedure, task, and lineage model, so data engineers and data scientists share one pipeline instead of handing off between tools.

## 2. Feature engineering: the Python & SQL behind the Feature Store

The features live in the Snowflake Feature Store as a **managed Feature View** (`MORTGAGE_FEATURE_VIEW` v5) keyed on the `LOAN_ENTITY` entity (`LOAN_ID`). A Feature View is just SQL; Snowflake keeps it fresh as an incrementally refreshed **Dynamic Table**.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode

fs = FeatureStore(session=session, database=DB, name=SCHEMA, default_warehouse=WH,
                  creation_mode=CreationMode.FAIL_IF_NOT_EXIST)

display(fs.list_entities().to_pandas())
fvs = fs.list_feature_views().to_pandas()
display(fvs[[c for c in ["NAME", "VERSION", "ENTITIES", "REFRESH_FREQ", "REFRESH_MODE", "SCHEDULING_STATE", "DESC"]
             if c in fvs.columns]])

**How this Feature View is built (example code).** Creating it is a few lines of Python: register an **Entity** (the business object and its join key), express the feature logic as a Snowpark DataFrame or SQL, wrap it in a **FeatureView** with a refresh policy, and register a version. Snowflake then materialises it as an incrementally refreshed Dynamic Table.

The cell below rebuilds `MORTGAGE_FEATURE_VIEW` exactly that way. `REGISTER_EXAMPLE_FV = False` keeps the live v5 untouched; instead, the cell proves the definition reproduces v5 row for row. Setting it to `True` registers the same logic as **v6**: any change to feature logic gets a new version, so models trained on v5 keep their lineage.

In [ ]:
from snowflake.ml.feature_store import Entity, FeatureView

REGISTER_EXAMPLE_FV = False   # True registers this definition as v6 (a second Dynamic Table, refreshed daily)

# 1. Entity: the business object the features describe; the join key is how features are looked up
loan_entity = Entity(name="LOAN_ENTITY", join_keys=["LOAN_ID"], desc="Features defined on a per loan level")

# 2. Feature logic: plain SQL (or a Snowpark DataFrame). Deterministic, no CURRENT_TIMESTAMP, so it refreshes incrementally
FEATURE_SQL = f"""
WITH base AS (
    SELECT LOAN_ID, COUNTY_NAME,
           DATEADD('day', 370, TO_TIMESTAMP(TS)) AS TIMESTAMP,      -- demo data shifted forward to look current
           LOAN_AMOUNT_000S * 1000               AS LOAN_AMOUNT,
           APPLICANT_INCOME_000S * 1000          AS INCOME
    FROM {SOURCE_TABLE}
)
SELECT LOAN_ID, TIMESTAMP,
       MONTH(TIMESTAMP)     AS MONTH,
       DAYOFYEAR(TIMESTAMP) AS DAY_OF_YEAR,
       DAYOFWEEK(TIMESTAMP) AS DOTW,
       LOAN_AMOUNT, INCOME,
       INCOME / LOAN_AMOUNT AS INCOME_LOAN_RATIO,
       AVG(INCOME::INT) OVER (PARTITION BY COUNTY_NAME)::INT AS MEAN_COUNTY_INCOME,
       (INCOME > MEAN_COUNTY_INCOME)::INT                    AS HIGH_INCOME_FLAG,
       AVG(LOAN_AMOUNT) OVER (PARTITION BY COUNTY_NAME ORDER BY TIMESTAMP
                              RANGE BETWEEN INTERVAL '30 DAYS' PRECEDING AND CURRENT ROW) AS AVG_THIRTY_DAY_LOAN_AMOUNT
FROM base"""

# 3. Feature View: entity + logic + refresh policy. timestamp_col enables point-in-time correct training joins
example_fv = FeatureView(
    name=FV_NAME,
    entities=[loan_entity],
    feature_df=session.sql(FEATURE_SQL),
    timestamp_col="TIMESTAMP",
    refresh_freq="1 day",          # managed: Snowflake keeps it fresh as a Dynamic Table (None = external, view-backed)
    refresh_mode="INCREMENTAL",    # only changed source rows are reprocessed on each refresh
    desc="Mortgage application features per loan",
).attach_feature_desc({
    "MONTH": "Month of loan",
    "DAY_OF_YEAR": "Day of calendar year of loan",
    "DOTW": "Day of the week of loan",
    "LOAN_AMOUNT": "Loan amount in $USD",
    "INCOME": "Household income in $USD",
    "INCOME_LOAN_RATIO": "Ratio of INCOME / LOAN_AMOUNT",
    "MEAN_COUNTY_INCOME": "Average household income aggregated at county level",
    "HIGH_INCOME_FLAG": "1 if household income is above MEAN_COUNTY_INCOME",
    "AVG_THIRTY_DAY_LOAN_AMOUNT": "Rolling 30-day average LOAN_AMOUNT in the county",
})

# 4. Register: the entity once, then a new version for every change to the feature logic
if REGISTER_EXAMPLE_FV:
    if "LOAN_ENTITY" not in [r["NAME"] for r in fs.list_entities().collect()]:
        fs.register_entity(loan_entity)
    new_fv = fs.register_feature_view(feature_view=example_fv, version="6", block=True)
    print(f"Registered {new_fv.name} v{new_fv.version} | refresh {new_fv.refresh_freq} | {new_fv.status}")
else:
    print("Not registered (REGISTER_EXAMPLE_FV = False). Checking this definition against the live v5:")

# Proof: the example definition reproduces the registered v5 Dynamic Table row for row
num_cols = [c for c in FEATURES]
check = q(f"""
SELECT (SELECT COUNT(*) FROM ({FEATURE_SQL})) AS EXAMPLE_ROWS,
       (SELECT COUNT(*) FROM {FV_TABLE})       AS V5_ROWS,
       COUNT(*)                                AS MATCHED_ON_LOAN_ID,
       COUNT_IF(NOT EQUAL_NULL(e.TIMESTAMP, v.TIMESTAMP)) AS TIMESTAMP_MISMATCHES,
       {', '.join(f'COUNT_IF(NOT EQUAL_NULL(ROUND(e.{c}, 6), ROUND(v.{c}, 6))) AS {c}_MISMATCHES' for c in num_cols)}
FROM ({FEATURE_SQL}) e JOIN {FV_TABLE} v USING (LOAN_ID)""").T.rename(columns={0: "value"})
display(check)

v = check["value"]
same = v["EXAMPLE_ROWS"] == v["V5_ROWS"] == v["MATCHED_ON_LOAN_ID"] and v.filter(like="_MISMATCHES").sum() == 0
print("PASS: the example code reproduces MORTGAGE_FEATURE_VIEW v5 exactly" if same
      else "DIFFERENT: see the mismatch counts above")

In [ ]:
mortgage_fv = fs.get_feature_view(FV_NAME, FV_VERSION)
print("Entity        :", [e.name for e in mortgage_fv.entities], "join keys", mortgage_fv.entities[0].join_keys)
print("Timestamp col :", mortgage_fv.timestamp_col, "(enables point-in-time correct joins)")
print("Refresh       :", mortgage_fv.refresh_freq, "| status", mortgage_fv.status)
print("\n--- SQL that defines the features ---\n")
print(mortgage_fv.query)

In [ ]:
# Business descriptions travel with the features (and show up in the Snowsight Feature Store UI)
pd.DataFrame(mortgage_fv.feature_descs.items(), columns=["feature", "description"])

In [ ]:
# Physical object: an incremental Dynamic Table, plus its recent refreshes
display(show(f"SHOW DYNAMIC TABLES LIKE '{FV_NAME}${FV_VERSION}' IN SCHEMA {DB}.{SCHEMA}")
        [["name", "target_lag", "refresh_mode", "scheduling_state", "rows"]])
display(fs.get_refresh_history(mortgage_fv).to_pandas().head(5))
display(fs.read_feature_view(mortgage_fv).limit(5).to_pandas())

**How these features are built (all inside the Feature View SQL):**

| Feature | Logic | Type |
|---|---|---|
| `MONTH`, `DAY_OF_YEAR`, `DOTW` | calendar parts of the application timestamp | row-level |
| `LOAN_AMOUNT`, `INCOME` | unit conversion from thousands | row-level |
| `INCOME_LOAN_RATIO` | `INCOME / LOAN_AMOUNT` | row-level ratio |
| `MEAN_COUNTY_INCOME`, `HIGH_INCOME_FLAG` | county-level window average and comparison | window aggregate |
| `AVG_THIRTY_DAY_LOAN_AMOUNT` | 30-day rolling window per county (`RANGE BETWEEN INTERVAL '30 DAYS' PRECEDING`) | time-window aggregate |

Because the Feature View has a timestamp column, training sets built from it (Section 5) use an **ASOF join**: every training row only sees feature values that existed at the time of that loan event, so there's no label leakage.

> **💡 Where Snowflake differentiates**
>
> * **Features are SQL, refreshed incrementally by Snowflake.** No Spark job to schedule, no cluster to size, no separate orchestrator for the feature pipeline. Other platforms usually need a materialization job plus a scheduler per feature table.
> * Point-in-time correctness is built into `generate_dataset()`; teams don't hand-write ASOF/time-travel joins.
> * One definition serves training (offline) and inference, so there's no training/serving skew from re-implementing features twice.

## 3. Data-quality monitoring on the feature store

The Feature View's Dynamic Table has **Data Metric Functions (DMFs)** attached with **expectations** (pass/fail thresholds). They run serverless on a schedule, here hourly, and write to a system results table. No separate DQ framework or job is involved.

In [ ]:
# What is being monitored, and on what schedule
dmfs = q(f"""
SELECT metric_name,
       ARRAY_TO_STRING(TRANSFORM(PARSE_JSON(ref_arguments), a -> a:name::STRING), ', ') AS arguments,
       schedule, schedule_status
FROM TABLE({DB}.INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
        REF_ENTITY_NAME => '{FV_TABLE}', REF_ENTITY_DOMAIN => 'table'))
ORDER BY metric_name""")
dmfs

In [ ]:
# Expectation status right now (pass/fail per rule)
expectations = q(f"""
SELECT expectation_name, metric_name,
       ARRAY_TO_STRING(TRANSFORM(PARSE_JSON(arguments), a -> a:name::STRING), ', ') AS arguments,
       expectation_expression, value, expectation_violated
FROM TABLE(SYSTEM$EVALUATE_DATA_QUALITY_EXPECTATIONS(REF_ENTITY_NAME => '{FV_TABLE}'))
ORDER BY expectation_violated DESC, expectation_name""")
expectations

In [ ]:
# Trend: hourly measurements from the serverless DQ engine
dq_trend = q(f"""
SELECT DATE_TRUNC('hour', measurement_time) AS hour, metric_name,
       ARRAY_TO_STRING(argument_names, ', ') AS arguments, value
FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
WHERE table_database = '{DB}' AND table_schema = '{SCHEMA}' AND table_name = '{FV_NAME}${FV_VERSION}'
  AND measurement_time > DATEADD('day', -3, CURRENT_TIMESTAMP())
ORDER BY hour""")
pivot = dq_trend.assign(metric=dq_trend.METRIC_NAME + "(" + dq_trend.ARGUMENTS.str.slice(0, 25) + ")") \
                .pivot_table(index="HOUR", columns="metric", values="VALUE", aggfunc="last")
pivot.tail(6).T

In [ ]:
# Drill from a metric to the offending rows: which loans are missing INCOME?
bad_rows = q(f"""
SELECT LOAN_ID, TIMESTAMP, LOAN_AMOUNT, INCOME
FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
        REF_ENTITY_NAME => '{FV_TABLE}',
        METRIC_NAME     => 'snowflake.core.null_count',
        ARGUMENT_NAME   => 'INCOME'))
LIMIT 5""")
print("Rows with NULL INCOME:", q(f"SELECT COUNT_IF(INCOME IS NULL) AS n FROM {FV_TABLE}").N[0])
bad_rows

**In the UI:** Catalog » Database Explorer » `ML_DEMO.PUBLIC.MORTGAGE_FEATURE_VIEW$5` » **Data Quality** tab shows these metrics, the expectation status, and trends. From there you can enable notifications and anomaly detection on the same DMFs.

How the DMFs were attached (one-time setup, already done):
```sql
ALTER TABLE ML_DEMO.PUBLIC."MORTGAGE_FEATURE_VIEW$5" SET DATA_METRIC_SCHEDULE = '60 MINUTE';
ALTER TABLE ML_DEMO.PUBLIC."MORTGAGE_FEATURE_VIEW$5"
  ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_PERCENT ON (INCOME)
  EXPECTATION income_mostly_populated (VALUE <= 20);
```

> **💡 Where Snowflake differentiates**
>
> * DQ rules attach **directly to the feature table** as metadata, run on **serverless compute**, and land in a queryable results table. Other platforms typically need a separate expectations library, a scheduled job to run it, and somewhere to store and visualize results.
> * `SYSTEM$DATA_METRIC_SCAN` goes from a failing metric to the exact failing rows in one call.
> * DQ results, features, models and monitors share one RBAC and one lineage graph (Section 5).

## 4. Existing model compatibility: bring a model trained elsewhere

Teams often already have models trained on SageMaker, Azure ML, Vertex, on-prem, or a laptop. If the model is a **natively supported type**, it goes into the Snowflake Model Registry as-is: no retraining, no format conversion, no wrapper code.

**Natively supported:** scikit-learn (incl. pipelines), XGBoost (sklearn API and `Booster`), LightGBM, CatBoost, PyTorch / TorchScript, TensorFlow / Keras, MLflow PyFunc, Hugging Face pipelines and sentence-transformers, Prophet, and Snowpark ML. Anything else (ONNX runtimes, statsmodels, R-exported scoring code via Python) uses a small `CustomModel` wrapper.

### 4a. Example: moving a SageMaker XGBoost model (illustrative, not executed)

SageMaker writes trained artifacts to S3 as `model.tar.gz`; for the built-in XGBoost algorithm the archive holds a single `xgboost-model` file. The code below shows the full import. It is guarded so it does not run in this demo account.

In [ ]:
RUN_SAGEMAKER_EXAMPLE = False   # illustrative only: needs AWS access and an existing SageMaker training job

if RUN_SAGEMAKER_EXAMPLE:
    import tarfile, xgboost as xgb
    from snowflake.ml.registry import Registry

    # --- 1. Locate the artifact SageMaker produced (run wherever you have AWS credentials) ---
    #   import boto3
    #   job = boto3.client("sagemaker").describe_training_job(TrainingJobName="mortgage-xgb-2026-10-01")
    #   job["ModelArtifacts"]["S3ModelArtifacts"]   -> s3://ml-artifacts/mortgage/output/model.tar.gz
    #   job["AlgorithmSpecification"]["TrainingImage"]  -> pins the xgboost version to reuse (e.g. 1.7-1)

    # --- 2. Read it into Snowflake through an external stage (no egress code, governed by a storage integration) ---
    session.sql("""
        CREATE STAGE IF NOT EXISTS ML_DEMO.PUBLIC.SAGEMAKER_ARTIFACTS
          URL = 's3://ml-artifacts/mortgage/output/'
          STORAGE_INTEGRATION = S3_ML_ARTIFACTS_INT""").collect()
    session.file.get("@ML_DEMO.PUBLIC.SAGEMAKER_ARTIFACTS/model.tar.gz", "/tmp/sagemaker/")
    with tarfile.open("/tmp/sagemaker/model.tar.gz") as tar:
        tar.extractall("/tmp/sagemaker/model")

    # --- 3. Load with the same library the model was trained with: no conversion ---
    booster = xgb.Booster()
    booster.load_model("/tmp/sagemaker/model/xgboost-model")
    # (Older SageMaker XGBoost containers pickled the Booster: pickle.load() it with the matching xgboost version.)

    # --- 4. Register it: now it is a governed, versioned, SQL-callable Snowflake object ---
    reg = Registry(session=session, database_name="ML_DEMO", schema_name="PUBLIC")
    mv = reg.log_model(
        booster,
        model_name="MORTGAGE_APPROVAL_SAGEMAKER_XGB",
        version_name="V1",
        sample_input_data=session.table("ML_DEMO.PUBLIC.SOME_FEATURE_SAMPLE").limit(100),  # sets signature + lineage
        conda_dependencies=["xgboost==1.7.6"],              # match the training container's version
        target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
        comment="Imported from SageMaker training job mortgage-xgb-2026-10-01",
    )
    # Then call it from SQL:  SELECT MORTGAGE_APPROVAL_SAGEMAKER_XGB!PREDICT(<features>) FROM ...

### 4b. Executed: import a model trained "elsewhere" and reproduce its baseline

To prove this end to end, we simulate an external team:

1. **"Elsewhere" training code** uses only pandas + xgboost (no Snowflake APIs). It trains a native `xgboost.Booster`, saves the same `xgboost-model` file SageMaker would produce, and records **baseline predictions** on a fixed holdout of 2,000 loans.
2. **Import:** load the file fresh, register it, and score the same holdout **inside a Snowflake warehouse**.
3. **Acceptance test:** predictions must match the baseline within **1e-6**, and AUC must be identical.

In [ ]:
# ---- Hand-off data: the extract the external team trained on (a pandas frame, like a CSV/Parquet export) ----
extract = (fs.generate_training_set(
               spine_df=session.sql(f"""SELECT LOAN_ID, DATEADD('day', 370, TO_TIMESTAMP(TS)) AS EVENT_TIMESTAMP, {LABEL}
                                        FROM {SOURCE_TABLE}"""),
               features=[mortgage_fv], spine_timestamp_col="EVENT_TIMESTAMP", spine_label_cols=[LABEL])
           .select(["LOAN_ID", *FEATURES, LABEL]).to_pandas()
           .sort_values("LOAN_ID").reset_index(drop=True))
extract[FEATURES] = extract[FEATURES].astype("float64")
holdout_mask = (extract["LOAN_ID"] % 10 == 0)
# Seeded random sample: the source is ordered by LOAN_ID, so .head() would yield a single-class holdout (AUC undefined)
ext_train = extract[~holdout_mask]
ext_holdout = extract[holdout_mask].sample(n=min(2000, int(holdout_mask.sum())), random_state=7).sort_values("LOAN_ID")
print(f"external training rows: {len(ext_train):,} | fixed holdout: {len(ext_holdout):,} | holdout label mix: {ext_holdout[LABEL].value_counts().to_dict()}")

In [ ]:
# ================= "ELSEWHERE" TRAINING CODE (no Snowflake imports) =================
import os, xgboost as xgb
from sklearn.metrics import roc_auc_score

def train_elsewhere(train_df, features, label):
    dtrain = xgb.DMatrix(train_df[features], label=train_df[label], feature_names=features)
    params = {"objective": "binary:logistic", "eval_metric": "auc", "max_depth": 5,
              "eta": 0.1, "subsample": 0.8, "colsample_bytree": 0.8, "seed": 7, "nthread": 4}
    return xgb.train(params, dtrain, num_boost_round=150)

booster_ext = train_elsewhere(ext_train, FEATURES, LABEL)
os.makedirs("/tmp/external_model", exist_ok=True)
ARTIFACT = "/tmp/external_model/xgboost-model"          # same artifact name SageMaker uses
booster_ext.save_model(ARTIFACT)

baseline_pred = booster_ext.predict(xgb.DMatrix(ext_holdout[FEATURES], feature_names=FEATURES))
baseline_auc = roc_auc_score(ext_holdout[LABEL], baseline_pred)
artifact_sha = hashlib.sha256(open(ARTIFACT, "rb").read()).hexdigest()
print(f"baseline holdout AUC = {baseline_auc:.6f} | artifact sha256 = {artifact_sha[:16]}…")
# ====================================================================================

In [ ]:
# ================= IMPORT INTO SNOWFLAKE =================
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)

def version_exists(model_name, version_name):
    try:
        reg.get_model(model_name).version(version_name)
        return True
    except Exception:
        return False

# Registry metadata is JSON, which cannot hold NaN
assert np.isfinite(baseline_auc), "baseline_auc is NaN - holdout must contain both classes"
import_metrics = {"baseline_holdout_auc": float(baseline_auc), "artifact_sha256": artifact_sha}

imported = xgb.Booster()
imported.load_model(ARTIFACT)                            # load the file exactly as handed over

if version_exists(EXTERNAL_MODEL_NAME, "V1"):
    mv_ext = reg.get_model(EXTERNAL_MODEL_NAME).version("V1")
    print("Already imported; reusing", EXTERNAL_MODEL_NAME, "V1")
    for k, v in import_metrics.items():                  # backfill metrics if a prior run left them unset
        if mv_ext.show_metrics().get(k) != v:
            mv_ext.set_metric(k, v)
else:
    mv_ext = reg.log_model(
        imported,
        model_name=EXTERNAL_MODEL_NAME,
        version_name="V1",
        sample_input_data=ext_holdout[FEATURES].head(100),
        conda_dependencies=[f"xgboost=={version('xgboost')}"],   # pin the training library version
        target_platforms=["WAREHOUSE"],
        metrics=import_metrics,
        comment="Externally trained xgboost Booster (SageMaker-style artifact) imported without conversion",
    )
print("metrics:", mv_ext.show_metrics())
print(mv_ext.show_functions()[0]["name"], "->", [i.name for i in mv_ext.show_functions()[0]["signature"].inputs][:4], "…")

In [ ]:
# Score the same holdout inside a Snowflake warehouse and compare to the external baseline
holdout_sp = session.create_dataframe(ext_holdout[["LOAN_ID", *FEATURES, LABEL]])
scored = mv_ext.run(holdout_sp, function_name="predict").to_pandas().sort_values("LOAN_ID")
pred_col = [c for c in scored.columns if c not in ["LOAN_ID", *FEATURES, LABEL]][0]
snowflake_pred = scored[pred_col].astype("float64").to_numpy()

max_abs_diff = float(np.max(np.abs(snowflake_pred - baseline_pred)))
snowflake_auc = roc_auc_score(ext_holdout[LABEL], snowflake_pred)
parity = pd.DataFrame({
    "check": ["max |prediction difference|", "holdout AUC (external)", "holdout AUC (in Snowflake)", "AUC difference"],
    "value": [f"{max_abs_diff:.2e}", f"{baseline_auc:.6f}", f"{snowflake_auc:.6f}", f"{abs(snowflake_auc - baseline_auc):.2e}"],
})
display(parity)
assert max_abs_diff <= 1e-6, "Imported model does not reproduce the external baseline"
print("PASS: imported model reproduces the external baseline (tolerance 1e-6)")

In [ ]:
# The imported model is now an ordinary SQL function too
q(f"""SELECT LOAN_ID, ROUND({EXTERNAL_MODEL_NAME}!PREDICT({", ".join(FEATURES)}):output_feature_0, 4) AS approval_probability
      FROM {FV_TABLE} LIMIT 5""")

In [ ]:
# Portability record: what changed (and what didn't) to move the model
pd.DataFrame([
    ("Model code",        "0 lines changed. Training code untouched; ~10 lines to load the artifact + log_model()"),
    ("Artifact format",   "Unchanged: native xgboost file, no ONNX/PMML/custom-flavor conversion"),
    ("Library",           f"xgboost {version('xgboost')} pinned via conda_dependencies (warehouse) / pip (SPCS)"),
    ("Runtime",           f"Python {platform.python_version()} recorded with the version; warehouse runtime resolves the same packages"),
    ("Inference surface", "SQL (MODEL!PREDICT), Python mv.run(), or a REST service: same artifact"),
    ("Governance",        "Model is a schema object: RBAC grants, tags, comments, metrics, lineage"),
    ("Baseline",          f"max |Δprob| = {max_abs_diff:.1e}, AUC {baseline_auc:.4f} reproduced"),
], columns=["Dimension", "Result"])

> **💡 Where Snowflake differentiates**
>
> * **No re-platforming tax:** the external artifact is registered as-is and becomes a SQL function. Other platforms often require repackaging into their own model flavor or a serving container before it can be used.
> * **The acceptance test runs where production data lives.** There's no export of holdout data to validate the port.
> * The same registry object serves **batch SQL, Python, and REST**, so there's no separate deployment per consumption pattern.

## 5. Model training: reproducible datasets, tracked experiments, lineage

Reproducibility rests on three recorded references:

| Reference | Snowflake object | What it pins |
|---|---|---|
| **Data** | `Dataset` (versioned, immutable snapshot) | exact rows + feature values, built point-in-time from the Feature Store |
| **Code** | experiment run params | training-function source hash, hyperparameters, seeds, split rule |
| **Environment** | experiment run params + model version | Python, xgboost, scikit-learn, snowflake-ml-python, runtime image |

### 5a. Create an immutable Dataset from the Feature Store

In [ ]:
from snowflake.ml.dataset import load_dataset

spine_df = session.sql(f"""
    SELECT LOAN_ID, DATEADD('day', 370, TO_TIMESTAMP(TS)) AS EVENT_TIMESTAMP, {LABEL}
    FROM {SOURCE_TABLE}""")

existing_versions = show(f"SHOW VERSIONS IN DATASET {DB}.{SCHEMA}.{DATASET_NAME}")["version"].tolist()
if DATASET_VERSION in existing_versions:
    dataset = load_dataset(session, f"{DB}.{SCHEMA}.{DATASET_NAME}", DATASET_VERSION)
    print(f"Dataset {DATASET_NAME} {DATASET_VERSION} already exists: loading the immutable snapshot")
else:
    dataset = fs.generate_dataset(
        name=DATASET_NAME, version=DATASET_VERSION,
        spine_df=spine_df, features=[mortgage_fv],
        spine_timestamp_col="EVENT_TIMESTAMP", spine_label_cols=[LABEL],
        desc="Point-in-time training snapshot from MORTGAGE_FEATURE_VIEW v5 (Section 5 demo)")
    print(f"Created Dataset {DATASET_NAME} {DATASET_VERSION}")

display(show(f"SHOW VERSIONS IN DATASET {DB}.{SCHEMA}.{DATASET_NAME}")[["version", "created_on", "comment"]])
print("Feature views captured in this Dataset:",
      [f"{v.name} v{v.version}" for v in fs.load_feature_views_from_dataset(dataset)])
print("Backed by immutable files:", dataset.read.files()[:2], "…")

In [ ]:
def load_training_frame(ds):
    """Dataset -> pandas, deterministic row order and split (LOAN_ID % 5 == 0 is the holdout)."""
    pdf = ds.read.to_pandas().sort_values("LOAN_ID").reset_index(drop=True)
    pdf[FEATURES] = pdf[FEATURES].astype("float64")
    pdf[LABEL] = pdf[LABEL].astype("int64")
    is_test = (pdf["LOAN_ID"] % 5 == 0)
    return pdf[~is_test], pdf[is_test]

train_pdf, test_pdf = load_training_frame(dataset)
print(f"train {len(train_pdf):,} rows | test {len(test_pdf):,} rows | approval rate {train_pdf[LABEL].mean():.3f}")

### 5b. Experiment run 1: train with every reference tracked

One training function is the single source of truth. Its source hash is logged so a later rerun can prove it executed the same code.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, f1_score
from snowflake.ml.experiment import ExperimentTracking

HYPERPARAMS = {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.1, "subsample": 0.8,
               "colsample_bytree": 0.8, "random_state": 42, "tree_method": "hist", "n_jobs": 4}

def train_mortgage_model(train_df, test_df, hyperparams):
    model = XGBClassifier(eval_metric="logloss", **hyperparams)
    model.fit(train_df[FEATURES], train_df[LABEL])
    proba = model.predict_proba(test_df[FEATURES])[:, 1]
    metrics = {"roc_auc": roc_auc_score(test_df[LABEL], proba),
               "accuracy": accuracy_score(test_df[LABEL], proba >= 0.5),
               "f1": f1_score(test_df[LABEL], proba >= 0.5),
               "log_loss": log_loss(test_df[LABEL], proba)}
    return model, proba, {k: float(v) for k, v in metrics.items()}

CODE_SHA = hashlib.sha256(inspect.getsource(train_mortgage_model).encode()).hexdigest()

def environment_refs():
    return {"python": platform.python_version(), "xgboost": version("xgboost"),
            "scikit_learn": version("scikit-learn"), "snowflake_ml_python": version("snowflake-ml-python"),
            "runtime_image": os_env("SNOWFLAKE_NOTEBOOK_RUNTIME_VERSION")}

def os_env(k):
    import os
    return os.environ.get(k, "local")

exp = ExperimentTracking(session=session, database_name=DB, schema_name=SCHEMA)
exp.set_experiment(EXPERIMENT_NAME)
RUN_1 = "TRAIN_" + time.strftime("%Y%m%d_%H%M%S")
print("experiment", EXPERIMENT_NAME, "| run", RUN_1, "| code sha", CODE_SHA[:12])

In [ ]:
with exp.start_run(RUN_1):
    exp.log_params({**HYPERPARAMS,
                    "data.dataset": f"{DB}.{SCHEMA}.{DATASET_NAME}", "data.dataset_version": DATASET_VERSION,
                    "data.features": ",".join(FEATURES), "data.label": LABEL, "data.split_rule": "LOAN_ID % 5 == 0 -> test",
                    "code.train_fn": "train_mortgage_model", "code.sha256": CODE_SHA,
                    **{f"env.{k}": v for k, v in environment_refs().items()}})
    model_1, proba_1, metrics_1 = train_mortgage_model(train_pdf, test_pdf, HYPERPARAMS)
    exp.log_metrics(metrics_1)

    # Register the model from inside the run: links run -> model, and Dataset -> model lineage
    if version_exists(MODEL_NAME, TRAIN_VERSION):
        mv_train = reg.get_model(MODEL_NAME).version(TRAIN_VERSION)
        print(f"{MODEL_NAME} {TRAIN_VERSION} already registered by an earlier run; reusing it")
    else:
        ds_sp = dataset.read.to_snowpark_dataframe()
        mv_train = exp.log_model(
            model_1, model_name=MODEL_NAME, version_name=TRAIN_VERSION,
            sample_input_data=ds_sp.select([ds_sp[c].cast("DOUBLE").alias(c) for c in FEATURES]),
            conda_dependencies=[f"xgboost=={version('xgboost')}"],
            target_platforms=["WAREHOUSE"],
            metrics=metrics_1,
            comment=f"Trained in experiment {EXPERIMENT_NAME} run {RUN_1} on Dataset {DATASET_NAME} {DATASET_VERSION}")
        print(f"Registered {MODEL_NAME} {TRAIN_VERSION}")

pd.Series(metrics_1, name=RUN_1).round(6)

### 5c. Rerun the recorded experiment and reproduce the result

The rerun reads **only what the tracker recorded** for run 1: dataset name and version, hyperparameters, code hash, and environment. It rebuilds everything from those references and checks the results against an agreed tolerance.

In [ ]:
TOLERANCE = 1e-6

recorded = show(f"SHOW RUN PARAMETERS IN EXPERIMENT {DB}.{SCHEMA}.{EXPERIMENT_NAME} RUN {RUN_1}")
rec = {"run_name": RUN_1, **dict(zip(recorded["name"], recorded["value"]))}

# 1) Environment + code references must match what was recorded
env_now = {f"env.{k}": v for k, v in environment_refs().items()}
checks = [(k, rec.get(k), v, rec.get(k) == v) for k, v in env_now.items()]
checks.append(("code.sha256", rec["code.sha256"][:16], CODE_SHA[:16], rec["code.sha256"] == CODE_SHA))
display(pd.DataFrame(checks, columns=["reference", "recorded", "now", "match"]))

# 2) Rebuild data + hyperparameters strictly from the recorded references
ds_again = load_dataset(session, rec["data.dataset"], rec["data.dataset_version"])
hp_again = {k: type(HYPERPARAMS[k])(rec[k]) for k in HYPERPARAMS}
train_again, test_again = load_training_frame(ds_again)

RUN_2 = "REPRO_" + RUN_1
with exp.start_run(RUN_2):
    exp.log_params({**{k: rec[k] for k in rec if k != "run_name"}, "reproduces_run": RUN_1})
    model_2, proba_2, metrics_2 = train_mortgage_model(train_again, test_again, hp_again)
    exp.log_metrics(metrics_2)

In [ ]:
# 3) Compare the tracked metrics of both runs, plus the registered model, within tolerance
import json
# ExperimentTracking has no read API, so pull logged metrics from SHOW RUNS metadata
runs_df = session.sql(f"SHOW RUNS IN EXPERIMENT {DB}.{SCHEMA}.{EXPERIMENT_NAME}").to_pandas()
runs_df.columns = [c.strip('"') for c in runs_df.columns]
runs_df = runs_df[runs_df["name"].isin([RUN_1, RUN_2])]
tracked = pd.DataFrame({r["name"]: {k: v["value"] for k, v in json.loads(r["metadata"]).get("metrics", {}).items()}
                        for _, r in runs_df.iterrows()})[[RUN_1, RUN_2]]
tracked["abs_diff"] = (tracked[RUN_1] - tracked[RUN_2]).abs()
tracked["within_tolerance"] = tracked["abs_diff"] <= TOLERANCE
display(tracked)

registered_pred = (mv_train.run(session.create_dataframe(test_again[["LOAN_ID", *FEATURES]]), function_name="predict_proba")
                   .to_pandas().sort_values("LOAN_ID")["output_feature_1"].astype("float64").to_numpy())
reg_diff = np.max(np.abs(proba_2 - registered_pred))
print(f"max |Δprob| run 1 vs run 2            : {np.max(np.abs(proba_1 - proba_2)):.2e}")
print(f"max |Δprob| rerun vs registered {TRAIN_VERSION}: {reg_diff:.2e}")
assert tracked["within_tolerance"].all() and np.max(np.abs(proba_1 - proba_2)) <= TOLERANCE

# The registered version is only comparable if it was logged by RUN_1 (cell 5b reuses an existing version)
if RUN_1 in (mv_train.comment or ""):
    assert reg_diff <= TOLERANCE
    print(f"PASS: run {RUN_2} reproduces {RUN_1} and the registered model within {TOLERANCE}")
else:
    print(f"PASS: run {RUN_2} reproduces {RUN_1} within {TOLERANCE}")
    print(f"NOTE: {MODEL_NAME} {TRAIN_VERSION} was registered by an earlier run ({mv_train.comment}); "
          f"register a new version from {RUN_1} to check registry parity")

**In the UI:** **AI & ML » Experiments » `MORTGAGE_APPROVAL_EXPERIMENT`**. Select both runs to compare parameters and metrics side by side; the run's **Models** tab links to the registered version.

### 5d. Lineage: data → Feature Store → Dataset → Model (→ Service)

Lineage was captured automatically by the API calls above (`generate_dataset` and `log_model` with Dataset-backed `sample_input_data`). Nothing was registered by hand.

In [ ]:
# In code (Python API)
upstream = list(mv_train.lineage(direction="upstream"))
for node in upstream:
    print(f"  {MODEL_NAME} {TRAIN_VERSION}  ◄──  {node._lineage_node_domain:13s} {node._lineage_node_name}"
          + (f"  (version {node._lineage_node_version})" if getattr(node, "_lineage_node_version", None) else ""))
print()
for node in dataset.lineage(direction="downstream", domain_filter={"model"}):
    print(f"  {DATASET_NAME} {DATASET_VERSION}  ──►  model {node._lineage_node_name} {getattr(node, '_lineage_node_version', '')}")

# The experiment node's version is the run that logged this model version (ExperimentTracking has no read API in 1.31)
logged_by = next((n._lineage_node_version for n in upstream if n._lineage_node_domain == "experiment"), None)
print(f"\n{MODEL_NAME} {TRAIN_VERSION} was logged by run {logged_by}"
      + ("" if logged_by == RUN_1 else f"; this session's run {RUN_1} reused it instead of registering a duplicate"))

In [ ]:
# In SQL: the full chain, including the production model's downstream inference service
lineage = q(f"""
SELECT 'upstream of {MODEL_NAME} {TRAIN_VERSION}' AS query, distance,
       source_object_domain || ' ' || source_object_name || COALESCE(' v' || source_object_version, '') AS source,
       target_object_domain || ' ' || target_object_name || COALESCE(' v' || target_object_version, '') AS target
FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE('{DB}.{SCHEMA}.{MODEL_NAME}', 'MODEL', 'upstream', 3, '{TRAIN_VERSION}'))
UNION ALL
SELECT 'downstream of {MODEL_NAME} {SERVING_VERSION}', distance,
       source_object_domain || ' ' || source_object_name || COALESCE(' v' || source_object_version, ''),
       target_object_domain || ' ' || target_object_name || COALESCE(' v' || target_object_version, '')
FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE('{DB}.{SCHEMA}.{MODEL_NAME}', 'MODEL', 'downstream', 2, '{SERVING_VERSION}'))
ORDER BY 1, 2""")
lineage

**In the UI:**
* **AI & ML » Models » `MORTGAGE_APPROVAL_CLASSIFIER` » V3 » Lineage** shows the graph `MORTGAGE_LENDING_DEMO_DATA → MORTGAGE_FEATURE_VIEW v5 → MORTGAGE_APPROVAL_TRAINING_DS V2 → MORTGAGE_APPROVAL_CLASSIFIER V3`.
* Select **V2** to see the same chain continue into the deployed service `MORTGAGE_APPROVAL_SERVICE`.
* From **Catalog » Database Explorer » MORTGAGE_LENDING_DEMO_DATA » Lineage**, the downstream view answers "which models were trained on this table?" (impact analysis before changing a source column).

> **💡 Where Snowflake differentiates**
>
> * **Reproducibility comes from objects, not conventions.** The Dataset is an immutable, versioned snapshot, so "rerun on the same data" doesn't depend on a table that kept changing or on time-travel retention windows.
> * **No tracking server to run.** Experiments, runs, params, metrics and model links are Snowflake objects with the same RBAC as the data. Other platforms commonly run (and secure, back up, upgrade) a separate tracking service.
> * **Lineage is a side effect of normal API use** and is queryable from SQL. There's no separate catalog registration, tagging step, or lineage agent to wire into training code.

## 6. Model inference: command line, latency, scaling, failure modes

One registered model, three ways to call it, all governed by the same RBAC:

| Path | Runs on | Best for | Call |
|---|---|---|---|
| **Warehouse (batch)** | Virtual warehouse | Scoring tables in SQL pipelines, dbt, Dynamic Tables | `MODEL(MORTGAGE_APPROVAL_CLASSIFIER, PRODUCTION)!PREDICT_PROBA(...)` |
| **Service function (SQL → SPCS)** | `MORTGAGE_APPROVAL_SERVICE` | SQL callers that want the always-on, scaled service | `MORTGAGE_APPROVAL_SERVICE!PREDICT_PROBA(...)` |
| **REST endpoint** | `MORTGAGE_APPROVAL_SERVICE` (public ingress) | Apps, APIs, the command line | `POST https://<ingress_url>/predict-proba` |

`MORTGAGE_APPROVAL_SERVICE` serves **V2** on compute pool `MORTGAGE_INFERENCE` (CPU_X64_S, 1–2 nodes).

In [ ]:
import requests, statistics, threading
import concurrent.futures as cf

MODEL_FQN   = f"{DB}.{SCHEMA}.{MODEL_NAME}"     # MODEL(<name>, <version|alias>) needs the qualified name
SERVICE_FQN = f"{DB}.{SCHEMA}.{SERVICE_NAME}"
DEPLOY_SERVICE_IF_MISSING = True    # only deploys when the service is missing (~10-20 min image build + SPCS credits)

mv_serving = reg.get_model(MODEL_NAME).version(SERVING_VERSION)

def service_exists():
    return not show(f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DB}.{SCHEMA}").empty

if not service_exists() and DEPLOY_SERVICE_IF_MISSING:
    t0 = time.time()
    # Snowflake builds the serving image from the registered model + its pinned dependencies, then starts the service
    mv_serving.create_service(service_name=SERVICE_FQN, service_compute_pool=COMPUTE_POOL,
                              ingress_enabled=True, min_instances=1, max_instances=2)
    print(f"Deployed {MODEL_NAME} {SERVING_VERSION} as {SERVICE_FQN} in {(time.time() - t0) / 60:.1f} min")

SERVICE_UP = service_exists()
INTERNAL_URL = None
if SERVICE_UP:
    try:
        session.sql(f"ALTER SERVICE {SERVICE_FQN} RESUME").collect()
    except Exception as e:                         # already running
        print("resume:", str(e).splitlines()[0])
    print("wait:", session.sql(f"SELECT SYSTEM$WAIT_FOR_SERVICES(900, '{SERVICE_FQN}')").collect()[0][0])
    display(show(f"SHOW ENDPOINTS IN SERVICE {SERVICE_FQN}")[["name", "port", "is_public", "ingress_url"]])
    svcs = mv_serving.list_services()
    INTERNAL_URL = svcs[svcs["name"].str.upper().str.endswith(SERVICE_NAME)]["internal_endpoint"].iloc[0]
    INTERNAL_URL = INTERNAL_URL if INTERNAL_URL.startswith("http") else f"http://{INTERNAL_URL}"
    print("internal endpoint (from inside Snowflake, no PAT needed):", INTERNAL_URL)
else:
    print(f"{SERVICE_FQN} is not deployed. Warehouse inference below still runs; the REST, scaling and "
          "failure-mode cells skip until DEPLOY_SERVICE_IF_MISSING = True is run once.")

### 6a. Call it from the command line

Run these in a local terminal. The PAT is read from the `password` field of the `[connections.<your_connection>]` connection in `~/.snowflake/config.toml` (requires Python 3.11+ for `tomllib`), so it never appears on screen or in shell history.

```bash
# 1. Token from ~/.snowflake/config.toml -> [connections.<your_connection>].password
export SNOWFLAKE_PAT=$(python3 -c "import tomllib, pathlib; print(tomllib.loads(pathlib.Path('~/.snowflake/config.toml').expanduser().read_text())['connections']['<your_connection>']['password'])")

# 2. Public endpoint of the service
export ENDPOINT=$(snow sql -c <your_connection> --format json \
  -q "SHOW ENDPOINTS IN SERVICE ML_DEMO.PUBLIC.MORTGAGE_APPROVAL_SERVICE" | jq -r '.[] | select(.name=="inference") | .ingress_url')

# 3. One prediction, with HTTP status and wall-clock latency
curl -s -w "\nHTTP %{http_code}  total %{time_total}s\n" -X POST "https://$ENDPOINT/predict-proba" \
  -H "Authorization: Snowflake Token=\"$SNOWFLAKE_PAT\"" -H "Content-Type: application/json" \
  -d @payload_1row.json

# 4. Latency distribution over 50 sequential calls (p50 / p95 / max, seconds)
for i in $(seq 1 50); do
  curl -s -o /dev/null -w "%{time_total}\n" -X POST "https://$ENDPOINT/predict-proba" \
    -H "Authorization: Snowflake Token=\"$SNOWFLAKE_PAT\"" -H "Content-Type: application/json" -d @payload_1row.json
done | sort -n | awk '{a[NR]=$1} END {printf "p50 %.3fs  p95 %.3fs  max %.3fs\n", a[int(NR*0.5)], a[int(NR*0.95)], a[NR]}'

# 5. Failure mode: a bad token is rejected at Snowflake's ingress with HTTP 401, before it reaches the model
curl -s -o /dev/null -w "HTTP %{http_code}\n" -X POST "https://$ENDPOINT/predict-proba" \
  -H 'Authorization: Snowflake Token="not-a-real-token"' -H "Content-Type: application/json" -d @payload_1row.json

# 6. The same model from SQL via the CLI (service function -> SPCS)
snow sql -c <your_connection> -q "SELECT ML_DEMO.PUBLIC.MORTGAGE_APPROVAL_SERVICE!PREDICT_PROBA(10, 290, 3, 250000, 95000, 0.38, 108569, 0, 231500):output_feature_1 AS approval_probability"
```

The cell below writes `payload_1row.json` and `payload_100rows.json` from real holdout loans and stages them in `@ML_DEMO.PUBLIC.MORTGAGE_CLI`; copy them next to where you run `curl`. Payloads use the service's row format: `{"data": [[row_index, feature_1, …, feature_9], …]}`, with features in the model signature's order.

In [ ]:
sample_rows = test_pdf.head(100)[FEATURES].astype("float64").reset_index(drop=True)

def rest_payload(df):
    # ~14% of applications have no income: send JSON null (NaN is not valid JSON); the model treats it as missing
    return {"data": [[i, *(None if pd.isna(v) else v for v in row)]
                     for i, row in enumerate(df.itertuples(index=False, name=None))]}

payload_1  = rest_payload(sample_rows.head(1))
payload_100 = rest_payload(sample_rows)
for fname, body in [("payload_1row.json", payload_1), ("payload_100rows.json", payload_100)]:
    with open(f"/tmp/{fname}", "w") as f:
        json.dump(body, f, allow_nan=False)

# Stage them so they can be downloaded (Snowsight » Data » ML_DEMO.PUBLIC » Stages » MORTGAGE_CLI)
session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.{SCHEMA}.MORTGAGE_CLI").collect()
for fname in ["payload_1row.json", "payload_100rows.json"]:
    session.file.put(f"/tmp/{fname}", f"@{DB}.{SCHEMA}.MORTGAGE_CLI", auto_compress=False, overwrite=True)

print("Feature order:", FEATURES)
print("payload_1row.json =", json.dumps(payload_1))
print(f"rows with a missing value sent as null: {sum(None in r for r in payload_100['data'])} of {len(payload_100['data'])}")
print(f"Download: snow stage copy @{DB}.{SCHEMA}.MORTGAGE_CLI/ . -c <your_connection>")

In [ ]:
def pctl(xs):
    xs = sorted(xs)
    return {"p50_ms": round(1000 * xs[len(xs) // 2], 1),
            "p95_ms": round(1000 * xs[max(int(len(xs) * 0.95) - 1, 0)], 1), "max_ms": round(1000 * xs[-1], 1)}

row = sample_rows.iloc[0]
sql_args = ", ".join(f"{float(row[c])}::FLOAT" for c in FEATURES)
latency = []

def record(surface, compute, call, rows_per_call, calls, stats):
    # Explicit columns (not a DataFrame index) so the surface shows up in Snowsight's table view
    latency.append({"surface": surface, "runs on": compute, "how it is called": call,
                    "rows per call": rows_per_call, "calls": calls, **stats})

# Warehouse: each call is a full SQL query (compile + run); this is the batch path, not the online one
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()
t = []
for _ in range(10):
    t0 = time.perf_counter()
    session.sql(f"SELECT MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA({sql_args}):output_feature_1").collect()
    t.append(time.perf_counter() - t0)
record("Warehouse SQL (single row)", f"warehouse {WH}", "SELECT MODEL(..., PRODUCTION)!PREDICT_PROBA(...)", 1, len(t), pctl(t))

# Warehouse batch: the whole holdout from the feature view in one statement
holdout_sql = f"FROM {FV_TABLE} WHERE LOAN_ID % 5 = 0 AND INCOME IS NOT NULL"
t0 = time.perf_counter()
session.sql(f"""SELECT COUNT_IF(MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA(
    {', '.join(f'{c}::FLOAT' for c in FEATURES)}):output_feature_1::FLOAT >= 0.5) {holdout_sql}""").collect()
wh_batch = time.perf_counter() - t0
rows_scored = session.sql(f"SELECT COUNT(*) {holdout_sql}").collect()[0][0]
record("Warehouse SQL (batch)", f"warehouse {WH}", "SELECT MODEL(..., PRODUCTION)!PREDICT_PROBA(...) FROM <table>",
       rows_scored, 1, {"p50_ms": round(1000 * wh_batch, 1), "p95_ms": None, "max_ms": None})

if SERVICE_UP:
    t = []
    for _ in range(10):
        t0 = time.perf_counter()
        session.sql(f"SELECT {SERVICE_FQN}!PREDICT_PROBA({sql_args}):output_feature_1").collect()
        t.append(time.perf_counter() - t0)
    record("Service function (SQL -> SPCS)", f"SPCS service on {COMPUTE_POOL}",
           f"SELECT {SERVICE_NAME}!PREDICT_PROBA(...)", 1, len(t), pctl(t))

    http = requests.Session()
    http.post(f"{INTERNAL_URL}/predict-proba", json=payload_1, timeout=30)          # warm-up
    t = []
    for _ in range(50):
        t0 = time.perf_counter()
        r = http.post(f"{INTERNAL_URL}/predict-proba", json=payload_1, timeout=30)
        t.append(time.perf_counter() - t0)
        r.raise_for_status()
    record("REST endpoint (single row)", f"SPCS service on {COMPUTE_POOL}", "POST /predict-proba (internal URL)", 1, len(t), pctl(t))
    t = []
    for _ in range(20):
        t0 = time.perf_counter()
        http.post(f"{INTERNAL_URL}/predict-proba", json=payload_100, timeout=30).raise_for_status()
        t.append(time.perf_counter() - t0)
    record("REST endpoint (100-row batch)", f"SPCS service on {COMPUTE_POOL}", "POST /predict-proba (internal URL)",
           len(payload_100["data"]), len(t), pctl(t))
    print("sample REST response:", r.json())
else:
    print("Service not deployed: only the warehouse paths were measured.")

session.sql("ALTER SESSION UNSET USE_CACHED_RESULT").collect()
pd.DataFrame(latency)

### 6b-i. Incremental, declarative inference with Dynamic Tables

With Dynamic Tables, you can process only changed rows from upstream sources. That processing can include running ML model inference. Here the input is the existing feature view (`MORTGAGE_FEATURE_VIEW` v5, itself a Dynamic Table), and the model is called through the **`PRODUCTION` alias**, so the pipeline never hard-codes a version number:

* **SQL:** the same idea as a plain Dynamic Table, `MORTGAGE_BATCH_INFERENCE_TABLE`.
* **Python (Feature Store):** register the predictions as their own feature view, `MORTGAGE_APPROVAL_PREDICTIONS`, keyed on `LOAN_ENTITY`. They are discoverable in the Feature Store UI and carry lineage back to the features.

Both refresh **incrementally**: when new loans land in the feature view, only those rows are scored. Target lag is `1 day` because a Dynamic Table can't be fresher than its source (the feature view refreshes daily).

**Aliases and incremental refresh.** The alias is resolved on each refresh, so after a promotion (Section 8) **new** rows are scored by the new version, while rows already in the table keep their earlier score. To re-score everything with the new production version, recreate the table (`CREATE OR REPLACE`, or re-register the feature view with `overwrite=True`).

Also check out the [documentation](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql#continuous-model-inference-with-dynamic-tables) to learn more.

In [ ]:
CREATE OR ALTER DYNAMIC TABLE ML_DEMO.PUBLIC.MORTGAGE_BATCH_INFERENCE_TABLE
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 day'           -- can't be fresher than its source, MORTGAGE_FEATURE_VIEW$5 (1 day)
    REFRESH_MODE = INCREMENTAL
    INITIALIZE = ON_CREATE
    COMMENT = 'Dynamic table with continuously updated model predictions (PRODUCTION alias)'
AS
SELECT
    LOAN_ID,
    TIMESTAMP,
    LOAN_AMOUNT,
    INCOME,
    MODEL(ML_DEMO.PUBLIC.MORTGAGE_APPROVAL_CLASSIFIER, PRODUCTION)!PREDICT_PROBA(
        MONTH::FLOAT, DAY_OF_YEAR::FLOAT, DOTW::FLOAT, LOAN_AMOUNT::FLOAT, INCOME::FLOAT,
        INCOME_LOAN_RATIO::FLOAT, MEAN_COUNTY_INCOME::FLOAT, HIGH_INCOME_FLAG::FLOAT,
        AVG_THIRTY_DAY_LOAN_AMOUNT::FLOAT
    ):output_feature_1::FLOAT AS APPROVAL_PROBABILITY
FROM ML_DEMO.PUBLIC."MORTGAGE_FEATURE_VIEW$5";

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureView

# Resolve the production model through its alias (never a hard-coded version)
prod_mv = Registry(session=session, database_name=DB, schema_name=SCHEMA).get_model(MODEL_NAME).version("PRODUCTION")
print(f"PRODUCTION alias -> {MODEL_NAME} {prod_mv.version_name}")

# Inference input: the existing feature view (LOAN_ID, TIMESTAMP + the 9 model features)
inference_dataset_sdf = fs.read_feature_view(mortgage_fv).select("LOAN_ID", "TIMESTAMP", *FEATURES)
inference_dataset_sdf.show(3)

# Ad-hoc scoring in the warehouse (add service_name=SERVICE_FQN to route through the SPCS service instead)
inference_result_sdf = prod_mv.run(inference_dataset_sdf.limit(10000), function_name="predict_proba")
inference_result_sdf.select("LOAN_ID", "TIMESTAMP", "output_feature_1").show(5)

## Create & register an inference Feature View: scheduled, incremental scoring through the PRODUCTION alias
inf_fv_name = "MORTGAGE_APPROVAL_PREDICTIONS"
inf_fv_version = "1"

# The SQL names the alias, so each refresh calls whichever version holds PRODUCTION at that moment
PROD_PROBA = (f"MODEL({DB}.{SCHEMA}.{MODEL_NAME}, PRODUCTION)!PREDICT_PROBA("
              f"{', '.join(f'{c}::FLOAT' for c in FEATURES)}):output_feature_1::FLOAT")
inference_features_sdf = session.sql(f"""
SELECT LOAN_ID, TIMESTAMP,
       {PROD_PROBA}                   AS APPROVAL_PROBABILITY,
       IFF({PROD_PROBA} >= 0.5, 1, 0) AS PREDICTED_APPROVAL
FROM {FV_TABLE}""")

# Adding per-feature descriptions to the FeatureView makes it easier to find features using Snowsight Universal Search.
inference_features_desc = {
    "APPROVAL_PROBABILITY": "Approval probability from the PRODUCTION version of MORTGAGE_APPROVAL_CLASSIFIER",
    "PREDICTED_APPROVAL": "1 if APPROVAL_PROBABILITY >= 0.5, else 0",
}

fv_inference_result = FeatureView(
    name=inf_fv_name,
    entities=[fs.get_entity("LOAN_ENTITY")],
    feature_df=inference_features_sdf,
    timestamp_col="TIMESTAMP",
    refresh_mode="INCREMENTAL",        # only new or changed feature rows are scored on each refresh
    refresh_freq="1 day",              # can't be fresher than its source, MORTGAGE_FEATURE_VIEW (1 day)
    desc=f"Approval predictions from {MODEL_NAME} (PRODUCTION alias) over {FV_NAME} v{FV_VERSION}",
).attach_feature_desc(inference_features_desc)

fv_inference_result = fs.register_feature_view(
    feature_view=fv_inference_result,
    version=inf_fv_version,
    overwrite=True,                    # rerunnable; also the way to re-score all rows after a promotion
    block=True,
)

fs_serving_fviews = fs.list_feature_views().filter(F.col("NAME") == inf_fv_name).sort(F.col("VERSION").desc())
display(fs_serving_fviews.select("NAME", "VERSION", "REFRESH_FREQ", "REFRESH_MODE", "SCHEDULING_STATE").to_pandas())
fs.read_feature_view(fv_inference_result).limit(5).to_pandas()

### 6b-ii. Scaling across instances and load balancing

* **Horizontal scaling.** A model service runs between `MIN_INSTANCES` and `MAX_INSTANCES` instances. With `min < max` it **autoscales** on traffic and utilization, and new instances are triggered after about a minute of sustained load. Set at deploy time with `create_service(min_instances=…, max_instances=…)` or later with `ALTER SERVICE … SET MIN_INSTANCES/MAX_INSTANCES`.
* **Load balancing.** Callers always hit one endpoint (public ingress or internal DNS), and SPCS spreads requests across all READY instances. Callers never see instance addresses.
* **Capacity.** By default each instance takes a whole node, so `MAX_INSTANCES` must fit the compute pool's `MAX_NODES` (2 here).
* **High availability.** Snowflake recommends 50% over-provisioning or at least 3 instances, whichever is higher, for node-failure tolerance. For zone-failure tolerance, use `PLACEMENT_GROUP = 'DISTRIBUTED'` on the compute pool.

The next cell pins the service at 2 instances, sends concurrent traffic, and shows both instances serving it.

In [ ]:
def instances():
    return show(f"SHOW SERVICE INSTANCES IN SERVICE {SERVICE_FQN}")

def wait_for_ready(n, timeout_s=900):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        inst = instances()
        ready = int((inst["status"].str.upper() == "READY").sum()) if not inst.empty else 0
        if ready >= n:
            return inst
        time.sleep(15)
    raise TimeoutError(f"{n} READY instances not reached in {timeout_s}s")

def burst(n_requests=400, workers=16, payload=None):
    payload = payload or payload_1
    lat, errors = [], []
    def call(_):
        t0 = time.perf_counter()
        try:
            requests.post(f"{INTERNAL_URL}/predict-proba", json=payload, timeout=30).raise_for_status()
            lat.append(time.perf_counter() - t0)
        except Exception as e:
            errors.append(type(e).__name__)
    t0 = time.perf_counter()
    with cf.ThreadPoolExecutor(workers) as ex:
        list(ex.map(call, range(n_requests)))
    wall = time.perf_counter() - t0
    return {"requests": n_requests, "ok": len(lat), "errors": len(errors), "throughput_rps": round(len(lat) / wall, 1),
            **(pctl(lat) if lat else {})}

if SERVICE_UP:
    session.sql(f"ALTER SERVICE {SERVICE_FQN} SET MIN_INSTANCES = 2 MAX_INSTANCES = 2").collect()
    inst = wait_for_ready(2)
    display(inst[[c for c in ["instance_id", "status", "start_time"] if c in inst.columns]])

    burst_start = session.sql("SELECT CURRENT_TIMESTAMP()").collect()[0][0]
    scale_results = pd.DataFrame([{"surface": "REST endpoint (single row)", "runs on": f"SPCS service on {COMPUTE_POOL}",
                                   "instances": 2, "concurrent clients": 16, **burst(400, 16)}])
    display(scale_results)

    # Which instances did the work? CPU per instance from the event table (metrics land within ~1-2 minutes)
    time.sleep(90)
    try:
        display(q(f"""
        SELECT RESOURCE_ATTRIBUTES:"snow.service.container.instance"::STRING AS instance,
               RECORD:metric:name::STRING AS metric, ROUND(AVG(VALUE::FLOAT), 3) AS avg_value, COUNT(*) AS samples
        FROM SNOWFLAKE.TELEMETRY.EVENTS
        WHERE RECORD_TYPE = 'METRIC' AND TIMESTAMP >= DATEADD('minute', -1, '{burst_start}'::TIMESTAMP_LTZ)
          AND RESOURCE_ATTRIBUTES:"snow.service.name"::STRING = '{SERVICE_NAME}'
          AND RECORD:metric:name::STRING IN ('container.cpu.usage', 'container.memory.usage')
        GROUP BY 1, 2 ORDER BY 2, 1"""))
    except Exception as e:
        print("Event-table metrics not readable with this role:", str(e).splitlines()[0])
else:
    print("Skipped: service not deployed.")

### 6c. Failure modes

Each failure is triggered for real, and the cell records what the caller sees:

| # | Failure | Expected behaviour |
|---|---|---|
| 1 | **Bad payload:** a missing feature, or a string where a number is expected | 4xx/5xx with an error message. The service stays healthy and the next good request succeeds. |
| 2 | **Instance loss under load:** scale 2 → 1 while traffic flows | Requests drain to the surviving instance. The cell counts errors during the scale-down. |
| 3 | **Cold start:** call a suspended service | `AUTO_RESUME` wakes it. The first call waits for resume and model load; avoid this in production with `MIN_INSTANCES ≥ 1`. |
| 4 | **Bad or missing token** | `401` from Snowflake's ingress (curl step 5 in 6a). The request never reaches the container, so a bad caller costs the model nothing. |
| 5 | **Bad row in a SQL batch** | Warehouse inference fails the whole statement. Validate with `TRY_TO_DOUBLE` first so one bad row doesn't fail the batch. |

In [ ]:
failures = []
WH_SQL, SVC_SQL, REST = "Warehouse SQL", "Service function (SQL -> SPCS)", "REST endpoint"

# 5. Bad row in SQL batch inference (warehouse path, runs without the service)
bad_rows_sql = f"""
    WITH incoming AS (SELECT * FROM VALUES ('10','290','3','250000','95000','0.38','108569','0','231500'),
                                           ('10','291','4','250000','n/a','0.38','108569','0','231500')
                      AS v({', '.join(FEATURES)}))"""
try:
    session.sql(bad_rows_sql + f""" SELECT MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA(
        {', '.join(f'{c}::FLOAT' for c in FEATURES)}) FROM incoming""").collect()
    failures.append(("5. batch with a bad row", WH_SQL, "no error"))
except Exception as e:
    failures.append(("5. batch with a bad row", WH_SQL, "statement failed: " + str(e).splitlines()[-1][:110]))
safe = session.sql(bad_rows_sql + f""", validated AS (SELECT {', '.join(f'TRY_TO_DOUBLE({c}) AS {c}' for c in FEATURES)} FROM incoming)
    SELECT COUNT(*) AS rows_in, COUNT_IF(INCOME IS NULL) AS quarantined,
           MAX(IFF(INCOME IS NOT NULL, MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA({', '.join(FEATURES)}):output_feature_1::FLOAT, NULL)) AS scored_prob
    FROM validated""").collect()[0]
failures.append(("5b. same batch with TRY_TO_DOUBLE validation", WH_SQL,
                 f"{safe['ROWS_IN']} rows in, {safe['QUARANTINED']} quarantined, good row scored {safe['SCORED_PROB']:.3f}"))

if SERVICE_UP:
    url = f"{INTERNAL_URL}/predict-proba"
    # 1. Bad payloads
    for label, body in [("1a. missing feature (8 of 9 columns)", {"data": [[0, *payload_1["data"][0][1:-1]]]}),
                        ("1b. string where a number is expected", {"data": [[0, "ten", *payload_1["data"][0][2:]]]})]:
        r = requests.post(url, json=body, timeout=30)
        failures.append((label, REST, f"HTTP {r.status_code}: {r.text[:110]}"))
    r = requests.post(url, json=payload_1, timeout=30)
    failures.append(("1c. next good request", REST, f"HTTP {r.status_code} (service unaffected)"))

    # 2. Instance loss under load: shrink 2 -> 1 while a burst is running
    result = {}
    t = threading.Thread(target=lambda: result.update(burst(600, 16)))
    t.start(); time.sleep(3)
    session.sql(f"ALTER SERVICE {SERVICE_FQN} SET MIN_INSTANCES = 1 MAX_INSTANCES = 1").collect()
    t.join()
    failures.append(("2. scale 2 -> 1 during 600 requests", REST,
                     f"{result['ok']} ok, {result['errors']} errors, p95 {result.get('p95_ms')} ms"))

    # 3. Cold start: suspend, then call through SQL (AUTO_RESUME wakes it)
    session.sql(f"ALTER SERVICE {SERVICE_FQN} SUSPEND").collect()
    time.sleep(20)
    t0 = time.perf_counter()
    session.sql(f"SELECT {SERVICE_FQN}!PREDICT_PROBA({sql_args}):output_feature_1").collect()
    cold = time.perf_counter() - t0
    t0 = time.perf_counter()
    session.sql(f"SELECT {SERVICE_FQN}!PREDICT_PROBA({sql_args}):output_feature_1").collect()
    failures.append(("3. cold start (suspended -> auto-resume)", SVC_SQL,
                     f"first call {cold:.1f}s, next call {time.perf_counter() - t0:.2f}s"))
else:
    failures.append(("1-3. service failure modes", f"{REST} / {SVC_SQL}", "skipped: service not deployed"))

failures.append(("4. bad or missing token", f"{REST} (public ingress)", "HTTP 401 at the ingress (see curl step 5 in 6a)"))
pd.DataFrame(failures, columns=["failure", "surface", "what the caller sees"])

> **💡 Where Snowflake differentiates**
>
> * **One model object, three serving paths.** Batch SQL, service functions, and REST all resolve to the same registered version under the same RBAC. There's no export to a separate serving stack, and no second copy of the model to secure.
> * **The serving stack is managed.** Snowflake builds the image, runs the HTTP server, load-balances across instances, autoscales, and auto-resumes. On other platforms that typically means a container registry, Kubernetes, an ingress controller, and an autoscaler to operate.
> * **Failures are observable in the same place as everything else.** Service logs and metrics land in the account event table, queryable in SQL next to the model and monitor data (Section 7).

## 7. Monitoring: operational and model metrics, drift, alerting, retraining

The production version (**V2**, alias `PRODUCTION`) gets a **model monitor**. A monitor reads an inference log (features, scores, predictions, and ground truth once it arrives), aggregates it daily on serverless compute, and exposes the results in SQL and in the Snowsight dashboard (**AI & ML » Models » MORTGAGE_APPROVAL_CLASSIFIER » Monitors**).

| Metric family | Examples | Source |
|---|---|---|
| **Model quality** | Accuracy, precision, recall, F1, ROC AUC, overall and per segment | `MODEL_MONITOR_PERFORMANCE_METRIC` |
| **Drift** | PSI, Jensen-Shannon, Wasserstein, difference of means, vs. a training baseline | `MODEL_MONITOR_DRIFT_METRIC` |
| **Data / volume** | Row counts, nulls, min / max / avg per feature | `MODEL_MONITOR_STAT_METRIC` |
| **Custom business metrics** | Any numeric per-row column you add, e.g. dollars approved, dollars wrongly approved | `CUSTOM_METRIC_COLUMNS` + `MODEL_MONITOR_STAT_METRIC` |
| **Operational** | Latency, throughput, errors, CPU / memory per instance | Section 6 measurements + service metrics in the event table |

### 7a. Build the inference log, with simulated drift

The holdout loans are scored through the `PRODUCTION` alias and spread across the last 30 days. **In the last 7 days we simulate two problems at once:**

* **Data drift:** applicant incomes arrive about 40% lower, as if from an upstream unit change or a new channel. The derived features are recomputed from the drifted income.
* **Concept drift:** a quarter of the loans the model approves go bad.

Two **custom metric** columns carry the business view: `APPROVED_LOAN_AMOUNT` (dollars approved) and `MISAPPROVED_LOAN_AMOUNT` (dollars approved that should have been denied).

In [ ]:
INFERENCE_LOG = f"{DB}.{SCHEMA}.MORTGAGE_V2_INFERENCE_LOG"
BASELINE_TBL  = f"{DB}.{SCHEMA}.MORTGAGE_V2_BASELINE"
MONITOR_NAME  = f"{DB}.{SCHEMA}.MORTGAGE_APPROVAL_V2_MONITOR"
DRIFT_DAYS, INCOME_SHIFT, BAD_APPROVAL_PCT = 7, 0.6, 25

def log_columns(label_expr):
    """Shared projection: model features as FLOAT, score/class, actual, segment and custom business metrics."""
    return f"""
    SELECT LOAN_ID, PREDICTION_TS,
           {', '.join(f'{c}::FLOAT AS {c}' for c in FEATURES)},
           APPROVAL_SCORE, PRED AS PREDICTED_APPROVED, {label_expr} AS ACTUAL_APPROVED,
           CASE WHEN LOAN_AMOUNT < 150000 THEN 'SMALL' WHEN LOAN_AMOUNT < 400000 THEN 'MEDIUM' ELSE 'LARGE' END AS LOAN_SIZE_BAND,
           IFF(PRED = 1, LOAN_AMOUNT, 0)::FLOAT                          AS APPROVED_LOAN_AMOUNT,
           IFF(PRED = 1 AND {label_expr} = 0, LOAN_AMOUNT, 0)::FLOAT     AS MISAPPROVED_LOAN_AMOUNT"""

score_expr = f"""MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA(
                   {', '.join(f'{c}::FLOAT' for c in FEATURES)}):output_feature_1::FLOAT"""

# Inference log: holdout loans (LOAN_ID % 5 = 0) spread over the 30 days before today; the last DRIFT_DAYS are drifted
session.sql(f"""
CREATE OR REPLACE TABLE {INFERENCE_LOG}
COMMENT = 'Demo inference log for {MODEL_NAME} V2: 30 days of scored holdout loans, last {DRIFT_DAYS} days with simulated drift'
AS
WITH base AS (
  SELECT f.LOAN_ID, s.{LABEL} AS LABEL_RAW, f.MONTH, f.DAY_OF_YEAR, f.DOTW, f.LOAN_AMOUNT, f.MEAN_COUNTY_INCOME,
         f.AVG_THIRTY_DAY_LOAN_AMOUNT, f.INCOME AS INCOME_RAW,
         DATEADD('minute', -(1 + ABS(HASH(f.LOAN_ID)) % (30 * 1440 - 1)), CURRENT_DATE()::TIMESTAMP_NTZ) AS PREDICTION_TS
  FROM {FV_TABLE} f JOIN {SOURCE_TABLE} s ON s.LOAN_ID = f.LOAN_ID
  WHERE f.LOAN_ID % 5 = 0 AND f.INCOME IS NOT NULL
),
featured AS (   -- data drift: incomes shrink, and the features derived from income are recomputed
  SELECT b.*, PREDICTION_TS >= DATEADD('day', -{DRIFT_DAYS}, CURRENT_DATE()) AS DRIFT_WINDOW,
         INCOME_RAW * IFF(DRIFT_WINDOW, {INCOME_SHIFT}, 1.0)  AS INCOME,
         DIV0NULL(INCOME, LOAN_AMOUNT)                        AS INCOME_LOAN_RATIO,
         IFF(INCOME > MEAN_COUNTY_INCOME, 1, 0)               AS HIGH_INCOME_FLAG
  FROM base b
),
scored AS (
  SELECT f.*, {score_expr} AS APPROVAL_SCORE, IFF(APPROVAL_SCORE >= 0.5, 1, 0) AS PRED FROM featured f
),
labeled AS (    -- concept drift: a share of the loans the model approves go bad
  SELECT s.*, IFF(DRIFT_WINDOW AND PRED = 1 AND ABS(HASH(LOAN_ID, 'drift')) % 100 < {BAD_APPROVAL_PCT}, 0, LABEL_RAW) AS ACTUAL
  FROM scored s
)
{log_columns('ACTUAL')}
FROM labeled""").collect()

# Baseline: V2 scores on a sample of its training split, the "normal" the monitor compares against
session.sql(f"""
CREATE OR REPLACE TABLE {BASELINE_TBL}
COMMENT = 'Baseline for MORTGAGE_APPROVAL_V2_MONITOR: V2 scores on a training-split sample (pre-drift reference)'
AS
WITH scored AS (
  SELECT f.*, s.{LABEL} AS ACTUAL, DATEADD('day', -60, CURRENT_DATE())::TIMESTAMP_NTZ AS PREDICTION_TS,
         {score_expr} AS APPROVAL_SCORE, IFF(APPROVAL_SCORE >= 0.5, 1, 0) AS PRED
  FROM {FV_TABLE} f JOIN {SOURCE_TABLE} s ON s.LOAN_ID = f.LOAN_ID
  WHERE f.LOAN_ID % 5 <> 0 AND f.LOAN_ID % 10 = 1 AND f.INCOME IS NOT NULL
)
{log_columns('ACTUAL')}
FROM scored""").collect()

q(f"""
SELECT IFF(PREDICTION_TS >= DATEADD('day', -{DRIFT_DAYS}, CURRENT_DATE()), 'last {DRIFT_DAYS} days (drifted)', 'days 8-30 (normal)') AS window,
       COUNT(*) AS predictions, ROUND(AVG(INCOME)) AS avg_income, ROUND(AVG(APPROVAL_SCORE), 3) AS avg_score,
       ROUND(AVG(IFF(PREDICTED_APPROVED = ACTUAL_APPROVED, 1, 0)), 3) AS accuracy,
       ROUND(SUM(MISAPPROVED_LOAN_AMOUNT) / COUNT(DISTINCT PREDICTION_TS::DATE) / 1e6, 1) AS misapproved_musd_per_day
FROM {INFERENCE_LOG} GROUP BY 1 ORDER BY 1""")

### 7b. Create the model monitor, with segments and custom metrics

One SQL statement wires everything up: the version and function being monitored, the log and baseline, score and class columns (for ROC AUC and for accuracy / precision / recall / F1), a **segment** column (`LOAN_SIZE_BAND`), and the two **custom metric** columns. Snowflake refreshes it on the configured interval. There's no monitoring job or dashboard to build.

In [ ]:
session.sql(f"""
CREATE OR REPLACE MODEL MONITOR {MONITOR_NAME} WITH
    MODEL = {MODEL_FQN}
    VERSION = '{SERVING_VERSION}'
    FUNCTION = 'PREDICT_PROBA'
    SOURCE = {INFERENCE_LOG}
    BASELINE = {BASELINE_TBL}
    WAREHOUSE = {WH}
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TS
    ID_COLUMNS = ('LOAN_ID')
    PREDICTION_SCORE_COLUMNS = ('APPROVAL_SCORE')
    PREDICTION_CLASS_COLUMNS = ('PREDICTED_APPROVED')
    ACTUAL_CLASS_COLUMNS = ('ACTUAL_APPROVED')
    SEGMENT_COLUMNS = ('LOAN_SIZE_BAND')
    CUSTOM_METRIC_COLUMNS = ('APPROVED_LOAN_AMOUNT', 'MISAPPROVED_LOAN_AMOUNT')
    COMMENT = 'Production monitor for {MODEL_NAME} {SERVING_VERSION} (demo log with simulated drift in the last {DRIFT_DAYS} days)'
""").collect()

mon = show(f"SHOW MODEL MONITORS LIKE 'MORTGAGE_APPROVAL_V2_MONITOR' IN SCHEMA {DB}.{SCHEMA}")
display(mon[["name", "monitor_state", "model_task", "aggregation_window", "refresh_interval", "warehouse"]])

### 7c. Model metrics, drift and custom business metrics over time

All of these are plain table functions, so the same numbers back the Snowsight monitor dashboard, this notebook, a Streamlit app, or an alert condition.

In [ ]:
WINDOW = "DATEADD('day', -30, CURRENT_DATE()), CURRENT_DATE()"

perf = q("\nUNION ALL\n".join(f"""
    SELECT EVENT_TIMESTAMP::DATE AS day, '{m}' AS metric, METRIC_VALUE AS value
    FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC('{MONITOR_NAME}', '{m}', '1 DAY', {WINDOW}))"""
    for m in ["CLASSIFICATION_ACCURACY", "PRECISION", "RECALL", "F1_SCORE", "ROC_AUC"]))

drift = q("\nUNION ALL\n".join(f"""
    SELECT EVENT_TIMESTAMP::DATE AS day, '{c}' AS feature, METRIC_VALUE AS psi
    FROM TABLE(MODEL_MONITOR_DRIFT_METRIC('{MONITOR_NAME}', 'POPULATION_STABILITY_INDEX', '{c}', '1 DAY', {WINDOW}))"""
    for c in ["INCOME", "INCOME_LOAN_RATIO", "LOAN_AMOUNT", "APPROVAL_SCORE"]))

# Custom business metrics: the monitor aggregates the custom columns like any other numeric column
custom = q("\nUNION ALL\n".join(f"""
    SELECT EVENT_TIMESTAMP::DATE AS day, '{c}' AS metric, METRIC_VALUE / 1e6 AS musd
    FROM TABLE(MODEL_MONITOR_STAT_METRIC('{MONITOR_NAME}', 'SUM', '{c}', '1 DAY', {WINDOW}))"""
    for c in ["APPROVED_LOAN_AMOUNT", "MISAPPROVED_LOAN_AMOUNT"]))
volume = q(f"""SELECT EVENT_TIMESTAMP::DATE AS day, METRIC_VALUE AS predictions
    FROM TABLE(MODEL_MONITOR_STAT_METRIC('{MONITOR_NAME}', 'COUNT', 'APPROVAL_SCORE', '1 DAY', {WINDOW}))""")

# Segment view: accuracy for large loans only
seg = q(f"""SELECT EVENT_TIMESTAMP::DATE AS day, METRIC_VALUE AS accuracy_large_loans
    FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC('{MONITOR_NAME}', 'CLASSIFICATION_ACCURACY', '1 DAY', {WINDOW},
         '{{"SEGMENTS": [{{"column": "LOAN_SIZE_BAND", "value": "LARGE"}}]}}'))""")

# The current day is partial; keep complete days only
for df in (perf, drift, custom, volume, seg):
    df.drop(df[df["DAY"] >= pd.Timestamp.today().normalize().date()].index, inplace=True)

drift_start = pd.Timestamp.today().normalize() - pd.Timedelta(days=DRIFT_DAYS)
fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
for (m, g) in perf.groupby("METRIC"):
    axes[0, 0].plot(pd.to_datetime(g["DAY"]), g["VALUE"], marker=".", label=m)
axes[0, 0].plot(pd.to_datetime(seg["DAY"]), seg["ACCURACY_LARGE_LOANS"], "k--", label="ACCURACY (LARGE loans segment)")
axes[0, 0].set_title("Model quality (needs ground truth)"); axes[0, 0].legend(fontsize=7)
for (c, g) in drift.groupby("FEATURE"):
    axes[0, 1].plot(pd.to_datetime(g["DAY"]), g["PSI"], marker=".", label=c)
axes[0, 1].axhline(0.25, color="red", ls=":", label="PSI 0.25 = significant shift")
axes[0, 1].set_title("Drift vs training baseline (PSI)"); axes[0, 1].legend(fontsize=7)
for (c, g) in custom.groupby("METRIC"):
    axes[1, 0].plot(pd.to_datetime(g["DAY"]), g["MUSD"], marker=".", label=c)
axes[1, 0].set_title("Custom business metrics ($M per day)"); axes[1, 0].legend(fontsize=7)
axes[1, 1].bar(pd.to_datetime(volume["DAY"]), volume["PREDICTIONS"], color="#29B5E8")
axes[1, 1].set_title("Prediction volume per day")
for ax in axes.flat:
    ax.axvline(drift_start, color="grey", alpha=0.6); ax.tick_params(axis="x", rotation=30)
fig.suptitle(f"{MODEL_NAME} {SERVING_VERSION} monitor (grey line = start of simulated drift)")
plt.tight_layout(); plt.show()

summary = (perf.assign(window=np.where(pd.to_datetime(perf["DAY"]) >= drift_start, "drifted", "normal"))
               .pivot_table(index="METRIC", columns="window", values="VALUE", aggfunc="mean").round(3))
summary.loc["PSI (INCOME)"] = (drift[drift["FEATURE"] == "INCOME"]
    .assign(window=lambda d: np.where(pd.to_datetime(d["DAY"]) >= drift_start, "drifted", "normal"))
    .groupby("window")["PSI"].mean().round(3))
summary.loc["MISAPPROVED $M/day (custom)"] = (custom[custom["METRIC"] == "MISAPPROVED_LOAN_AMOUNT"]
    .assign(window=lambda d: np.where(pd.to_datetime(d["DAY"]) >= drift_start, "drifted", "normal"))
    .groupby("window")["MUSD"].mean().round(1))
summary[["normal", "drifted"]]

**Custom metrics: what's supported.** Any **numeric per-row** column listed in `CUSTOM_METRIC_COLUMNS` is tracked over time like a feature: `SUM`, `AVG`, `MIN`, `MAX`, `COUNT` and `COUNT_NULL` per aggregation window and per segment, plus drift against the baseline. That covers most business KPIs: dollars approved, expected loss, the cost of a false positive, margin, or minutes saved.

Ratios of sums, or cohort metrics that can't be written per row (for example "approval-rate parity between two groups"), are just SQL over the same inference log. The alert below uses exactly that pattern alongside the monitor's own functions.

### 7d. Operational metrics

* **Serving latency and throughput** were measured in Section 6 (`latency`, `scale_results`).
* **Service health per instance** (CPU, memory, GPU, network, request counts) is written automatically to the account event table. Query it with `<service>!SPCS_GET_METRICS()` or straight from `SNOWFLAKE.TELEMETRY.EVENTS`. Logs come from `<service>!SPCS_GET_LOGS()`. The same data is in **Monitoring » Services & jobs » MORTGAGE_APPROVAL_SERVICE » Metrics / Logs**.
* **Pipeline health** is covered too: the monitor's own state and refresh errors (`DESCRIBE MODEL MONITOR`), plus prediction volume and null counts (the chart above).

In [ ]:
desc = show(f"DESCRIBE MODEL MONITOR {MONITOR_NAME}")
print("monitor state     :", desc["monitor_state"].iloc[0])
print("aggregation status:", desc["aggregation_status"].iloc[0] if "aggregation_status" in desc else "n/a")

nulls = q(f"""SELECT '{FEATURES[4]}' AS column_name, SUM(METRIC_VALUE) AS nulls_last_30d
    FROM TABLE(MODEL_MONITOR_STAT_METRIC('{MONITOR_NAME}', 'COUNT_NULL', 'INCOME', '1 DAY', {WINDOW}))""")
display(nulls)

if service_exists():
    svc_metrics = q(f"""
        SELECT DATE_TRUNC('minute', TIMESTAMP) AS minute, INSTANCE_ID, METRIC_NAME, ROUND(AVG(VALUE), 3) AS value
        FROM TABLE({SERVICE_FQN}!SPCS_GET_METRICS())
        WHERE TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
          AND METRIC_NAME IN ('container.cpu.usage', 'container.memory.usage')
        GROUP BY 1, 2, 3 ORDER BY 1 DESC LIMIT 40""")
    display(svc_metrics)
else:
    print("Service metrics: skipped (service not deployed).")

if "latency" in globals():
    display(pd.DataFrame(latency).T.rename_axis("serving path (Section 6)"))

### 7e. Threshold breach → alert the model owner

An **alert** checks the monitor every morning. If income PSI exceeds **0.25** (a significant shift) or accuracy falls below **0.70** on any of the last 3 complete days, it calls `NOTIFY_MODEL_OWNER()`, which:

1. opens a ticket in `MODEL_REVIEW_QUEUE` (the hand-off into the review workflow in 7f), and
2. emails the owner (`OWNER_EMAIL`) through an email notification integration, with the metrics that tripped it.

The alert is created **suspended** and run once on demand with `EXECUTE ALERT`, so the breach fires now without leaving a schedule running.

In [ ]:
PSI_MAX, ACC_MIN = 0.25, 0.70
EMAIL_INTEGRATION = "LTO_DEMO_EMAIL_INT"   # email notification integration the alert owner (SYSADMIN) can use; OWNER_EMAIL is an allowed recipient
REVIEW_QUEUE = f"{DB}.{SCHEMA}.MODEL_REVIEW_QUEUE"
ALERT_NAME   = f"{DB}.{SCHEMA}.MORTGAGE_MODEL_DRIFT_ALERT"
LAST_3_DAYS  = "DATEADD('day', -3, CURRENT_DATE()), DATEADD('day', -1, CURRENT_DATE())"   # daily buckets, complete days only

session.sql(f"""
CREATE TABLE IF NOT EXISTS {REVIEW_QUEUE} (
  TICKET_ID NUMBER AUTOINCREMENT, OPENED_AT TIMESTAMP_NTZ, MODEL VARCHAR, VERSION VARCHAR, MONITOR VARCHAR,
  REASON VARCHAR, MAX_INCOME_PSI FLOAT, MIN_ACCURACY FLOAT, MISAPPROVED_USD_3D FLOAT,
  STATUS VARCHAR, ASSIGNED_TO VARCHAR, RESOLUTION VARCHAR, CLOSED_AT TIMESTAMP_NTZ)""").collect()

session.sql(f"""
CREATE OR REPLACE PROCEDURE {DB}.{SCHEMA}.NOTIFY_MODEL_OWNER()
RETURNS VARCHAR
LANGUAGE SQL
COMMENT = 'Opens a review ticket and emails the model owner when {MONITOR_NAME} breaches its thresholds'
AS
$$
DECLARE
  psi FLOAT; acc FLOAT; misapproved FLOAT; reason VARCHAR; subject VARCHAR; body VARCHAR;
BEGIN
  SELECT MAX(METRIC_VALUE) INTO :psi FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    '{MONITOR_NAME}', 'POPULATION_STABILITY_INDEX', 'INCOME', '1 DAY', {LAST_3_DAYS}));
  SELECT MIN(METRIC_VALUE) INTO :acc FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    '{MONITOR_NAME}', 'CLASSIFICATION_ACCURACY', '1 DAY', {LAST_3_DAYS}));
  SELECT SUM(METRIC_VALUE) INTO :misapproved FROM TABLE(MODEL_MONITOR_STAT_METRIC(
    '{MONITOR_NAME}', 'SUM', 'MISAPPROVED_LOAN_AMOUNT', '1 DAY', {LAST_3_DAYS}));

  reason := ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
              IFF(:psi > {PSI_MAX}, 'INCOME drift PSI ' || ROUND(:psi, 2) || ' > {PSI_MAX}', NULL),
              IFF(:acc < {ACC_MIN}, 'accuracy ' || ROUND(:acc, 3) || ' < {ACC_MIN}', NULL)), '; ');

  INSERT INTO {REVIEW_QUEUE} (OPENED_AT, MODEL, VERSION, MONITOR, REASON, MAX_INCOME_PSI, MIN_ACCURACY,
                              MISAPPROVED_USD_3D, STATUS, ASSIGNED_TO)
  VALUES (CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, '{MODEL_FQN}', '{SERVING_VERSION}', '{MONITOR_NAME}', :reason,
          :psi, :acc, :misapproved, 'OPEN', '{OWNER_EMAIL}');

  subject := '[ML ALERT] {MODEL_NAME} {SERVING_VERSION}: ' || :reason;
  body := 'Model monitor {MONITOR_NAME} breached its thresholds over the last 3 days.' || CHAR(10) || CHAR(10) ||
          '  Max INCOME PSI      : ' || ROUND(:psi, 3) || '  (threshold {PSI_MAX})' || CHAR(10) ||
          '  Min daily accuracy  : ' || ROUND(:acc, 3) || '  (threshold {ACC_MIN})' || CHAR(10) ||
          '  Misapproved loans   : $' || TO_VARCHAR(ROUND(:misapproved / 1e6, 1)) || 'M over 3 days (custom metric)' || CHAR(10) || CHAR(10) ||
          'A review ticket was opened in {REVIEW_QUEUE}.' || CHAR(10) ||
          'Dashboard: Snowsight > AI & ML > Models > {MODEL_NAME} > Monitors > MORTGAGE_APPROVAL_V2_MONITOR' || CHAR(10) ||
          'Runbook: triage drift -> retrain challenger -> review -> promote via PRODUCTION alias (notebook sections 7f and 8).';
  CALL SYSTEM$SEND_EMAIL('{EMAIL_INTEGRATION}', '{OWNER_EMAIL}', :subject, :body);
  RETURN :subject;
END;
$$""").collect()

ALERT_CONDITION = f"""EXISTS (
      SELECT 1 FROM TABLE(MODEL_MONITOR_DRIFT_METRIC('{MONITOR_NAME}', 'POPULATION_STABILITY_INDEX', 'INCOME', '1 DAY', {LAST_3_DAYS}))
      WHERE METRIC_VALUE > {PSI_MAX}
      UNION ALL
      SELECT 1 FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC('{MONITOR_NAME}', 'CLASSIFICATION_ACCURACY', '1 DAY', {LAST_3_DAYS}))
      WHERE METRIC_VALUE < {ACC_MIN})"""

session.sql(f"""
CREATE OR REPLACE ALERT {ALERT_NAME}
  WAREHOUSE = {WH}
  SCHEDULE = 'USING CRON 0 7 * * * America/Los_Angeles'
  COMMENT = 'Daily check of {MONITOR_NAME}: INCOME PSI > {PSI_MAX} or accuracy < {ACC_MIN}'
  IF ({ALERT_CONDITION})
  THEN CALL {DB}.{SCHEMA}.NOTIFY_MODEL_OWNER()""").collect()

# Alerts are created suspended; run it once now (it stays suspended afterwards)
fired_at = session.sql("SELECT CURRENT_TIMESTAMP()").collect()[0][0]
session.sql(f"EXECUTE ALERT {ALERT_NAME}").collect()

hist = pd.DataFrame()
for _ in range(24):
    time.sleep(10)
    hist = q(f"""SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, SQL_ERROR_MESSAGE
        FROM TABLE(INFORMATION_SCHEMA.ALERT_HISTORY(SCHEDULED_TIME_RANGE_START => '{fired_at}'::TIMESTAMP_LTZ))
        WHERE NAME = 'MORTGAGE_MODEL_DRIFT_ALERT' ORDER BY SCHEDULED_TIME DESC LIMIT 1""")
    if not hist.empty and hist["STATE"].iloc[0] in ("TRIGGERED", "CONDITION_FALSE", "CONDITION_FAILED",
                                                     "ACTION_FAILED", "FAILED"):
        break
display(hist)

# The alert's owner role needs the account-level EXECUTE ALERT privilege to run it. Without it, evaluate the
# identical condition and run the identical action here, so the owner is still notified during the demo.
if not hist.empty and "EXECUTE ALERT privilege" in str(hist["SQL_ERROR_MESSAGE"].iloc[0]):
    print(f"Scheduled alert needs: USE ROLE ACCOUNTADMIN; GRANT EXECUTE ALERT ON ACCOUNT TO ROLE {ROLE};")
    breached = session.sql(f"SELECT {ALERT_CONDITION} AS breached").collect()[0]["BREACHED"]
    print("condition breached:", breached)
    if breached:
        print("action:", session.call(f"{DB}.{SCHEMA}.NOTIFY_MODEL_OWNER"))

display(show(f"SHOW ALERTS LIKE 'MORTGAGE_MODEL_DRIFT_ALERT' IN SCHEMA {DB}.{SCHEMA}")[["name", "state", "schedule"]])
q(f"SELECT TICKET_ID, OPENED_AT, VERSION, REASON, ROUND(MAX_INCOME_PSI, 3) AS MAX_INCOME_PSI, ROUND(MIN_ACCURACY, 3) AS MIN_ACCURACY, ROUND(MISAPPROVED_USD_3D / 1e6, 1) AS MISAPPROVED_MUSD_3D, STATUS, ASSIGNED_TO FROM {REVIEW_QUEUE} ORDER BY TICKET_ID DESC LIMIT 3")

### 7f. Retraining and review workflow

The alert starts a human-in-the-loop workflow. Every step is a Snowflake object, so each hand-off is governed and auditable:

| Step | Who | What happens | Snowflake object |
|---|---|---|---|
| 1. **Detect** | Automated | The monitor aggregates daily; the alert evaluates the thresholds | `MORTGAGE_APPROVAL_V2_MONITOR`, `MORTGAGE_MODEL_DRIFT_ALERT` |
| 2. **Notify and ticket** | Automated | The owner is emailed and a ticket opened | `NOTIFY_MODEL_OWNER()`, `MODEL_REVIEW_QUEUE` |
| 3. **Triage** | Model owner | Data drift, concept drift, or a pipeline bug? Check DQ results (Section 3), drift per feature and segment (7c), and lineage back to source (5d) | Monitor dashboard, `DATA_METRIC_SCAN`, `GET_LINEAGE` |
| 4. **Retrain** | Automated (on approval) | Snapshot fresh features into a **new Dataset version**, retrain with the tracked hyperparameters, and log a new model version tagged `CHALLENGER` with its metrics | `generate_dataset`, `ExperimentTracking`, `log_model` |
| 5. **Review** | Reviewer (separate role) | Compare challenger vs production on the same recent window: metrics, segments, custom $ metrics | Registry metrics, side-by-side scoring, optional shadow traffic via a Gateway |
| 6. **Promote or reject** | Promoter role | Move the `PRODUCTION` alias (Section 8), close the ticket, and create a monitor on the new version | `ALTER MODEL … SET ALIAS`, `MODEL_REVIEW_QUEUE` |
| 7. **Roll back** | Promoter role | Move the alias back; no redeploy for SQL callers | `ALTER MODEL … SET ALIAS` |

Steps 4 and 5 are automated below as a task graph. It's illustrative: created only if `RUN_RETRAIN_PIPELINE_EXAMPLE = True`, and left suspended so it never runs on its own.

In [ ]:
RUN_RETRAIN_PIPELINE_EXAMPLE = False   # creates RETRAIN_MORTGAGE_MODEL() and a suspended retrain -> evaluate task graph

RETRAIN_PROC_DDL = f"""
CREATE OR REPLACE PROCEDURE {DB}.{SCHEMA}.RETRAIN_MORTGAGE_MODEL(NEW_VERSION VARCHAR)
RETURNS OBJECT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python', 'xgboost', 'scikit-learn')
HANDLER = 'run'
AS
$$
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.registry import Registry
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier

FEATURES = {FEATURES}
HYPERPARAMS = {HYPERPARAMS}

def run(session, new_version):
    fs = FeatureStore(session, database='{DB}', name='{SCHEMA}', default_warehouse='{WH}',
                      creation_mode=CreationMode.FAIL_IF_NOT_EXIST)
    fv = fs.get_feature_view('{FV_NAME}', '{FV_VERSION}')
    spine = session.sql("SELECT LOAN_ID, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS EVENT_TIMESTAMP, {LABEL} FROM {SOURCE_TABLE}")
    # 1. Fresh, immutable training snapshot -> lineage source -> Dataset -> new model version
    ds = fs.generate_dataset(name='{DATASET_NAME}', version=new_version, spine_df=spine, features=[fv],
                             spine_timestamp_col='EVENT_TIMESTAMP', spine_label_cols=['{LABEL}'])
    pdf = ds.read.to_pandas().sort_values('LOAN_ID').dropna(subset=FEATURES)
    test = pdf['LOAN_ID'] % 5 == 0
    model = XGBClassifier(eval_metric='logloss', **HYPERPARAMS).fit(pdf.loc[~test, FEATURES], pdf.loc[~test, '{LABEL}'])
    proba = model.predict_proba(pdf.loc[test, FEATURES])[:, 1]
    y = pdf.loc[test, '{LABEL}']
    metrics = {{'roc_auc': float(roc_auc_score(y, proba)), 'accuracy': float(accuracy_score(y, proba >= 0.5)),
               'f1': float(f1_score(y, proba >= 0.5))}}
    # 2. Register as a candidate; production callers are unaffected until the alias moves
    reg = Registry(session, database_name='{DB}', schema_name='{SCHEMA}')
    mv = reg.log_model(model, model_name='{MODEL_NAME}', version_name=new_version,
                       sample_input_data=ds.read.to_snowpark_dataframe().select(FEATURES).limit(100),
                       metrics=metrics, target_platforms=['WAREHOUSE'],
                       comment=f'Retrained after drift alert on Dataset {DATASET_NAME} {{new_version}}')
    return {{'version': new_version, **metrics}}
$$"""

TASK_GRAPH_DDL = [
    f"""CREATE OR REPLACE TASK {DB}.{SCHEMA}.MORTGAGE_RETRAIN_TASK WAREHOUSE = {WH}
        COMMENT = 'Started by the model owner after triage; never scheduled'
        AS CALL {DB}.{SCHEMA}.RETRAIN_MORTGAGE_MODEL('V' || TO_VARCHAR(CURRENT_DATE(), 'YYYYMMDD'))""",
    f"""CREATE OR REPLACE TASK {DB}.{SCHEMA}.MORTGAGE_EVALUATE_CHALLENGER_TASK WAREHOUSE = {WH}
        AFTER {DB}.{SCHEMA}.MORTGAGE_RETRAIN_TASK
        AS UPDATE {REVIEW_QUEUE} SET STATUS = 'CHALLENGER_READY_FOR_REVIEW' WHERE STATUS = 'OPEN'""",
]

if RUN_RETRAIN_PIPELINE_EXAMPLE:
    session.sql(RETRAIN_PROC_DDL).collect()
    for ddl in TASK_GRAPH_DDL:
        session.sql(ddl).collect()
    # The owner kicks it off after triage:  EXECUTE TASK ML_DEMO.PUBLIC.MORTGAGE_RETRAIN_TASK
    display(show(f"SHOW TASKS LIKE 'MORTGAGE_%' IN SCHEMA {DB}.{SCHEMA}")[["name", "state", "predecessors"]])
else:
    print("Illustrative: set RUN_RETRAIN_PIPELINE_EXAMPLE = True to create the retrain procedure and task graph.")
    print("\n".join(RETRAIN_PROC_DDL.splitlines()[:12]), "\n…")

> **💡 Where Snowflake differentiates**
>
> * **Monitoring is a SQL object, not a project.** One `CREATE MODEL MONITOR` gives drift, quality, segment, volume, and **custom business metrics**, computed on serverless compute, with a built-in dashboard. Other platforms usually need a monitoring library, a scheduled job, a metrics store, and a dashboarding tool.
> * **Business KPIs sit next to ML metrics.** Dollars wrongly approved is tracked, segmented, and alerted on exactly like accuracy, so the conversation with risk owners uses their numbers.
> * **Detection to action stays in one platform.** The alert, email, ticket table, retraining procedure, and task graph share the same RBAC and audit trail as the data and the model, with no webhook glue between separate systems.

## 8. Promote and roll back with model aliases

Versions are immutable (`V1`, `V2`, `V3`, …). An **alias** is a movable label on exactly one version, and promotion means **moving the label**, not changing callers:

| Reference | Resolves to | Use it for |
|---|---|---|
| `MODEL(m, V3)!PREDICT_PROBA(...)` | Exactly V3, forever | Reproducibility, audits, tests |
| `MODEL(m, PRODUCTION)!PREDICT_PROBA(...)` | Whichever version holds the `PRODUCTION` alias **now** | Every production caller: SQL pipelines, dbt, Dynamic Tables, apps |
| `MODEL(m, CHALLENGER)!PREDICT_PROBA(...)` | The candidate under review | Side-by-side evaluation, shadow scoring |
| `m!PREDICT_PROBA(...)` | The model's `DEFAULT` version (`ALTER MODEL … SET DEFAULT_VERSION`) | The simplest dev/prod scheme |

Built-in system aliases also exist: `DEFAULT`, `FIRST` and `LAST`. A version holds at most one custom alias, and each alias name is unique within the model. Moving an alias requires **OWNERSHIP** of the model (Section 9), so promotion is a privileged, audited action.

The flow below is: current state → review challenger vs production on the drifted window → promote V3 → identical query now answers with V3 → roll back.

In [ ]:
def assign_aliases(mapping):
    """mapping: {version: alias}. Clears aliases on the affected versions and on any other holder of the alias first."""
    vers = show(f"SHOW VERSIONS IN MODEL {MODEL_FQN}")
    holders = {a: r["name"] for _, r in vers.iterrows() for a in json.loads(r["aliases"] or "[]")
               if a not in ("DEFAULT", "FIRST", "LAST")}
    to_clear = set(mapping) | {holders[a] for a in mapping.values() if a in holders}
    for v in to_clear:
        if any(holders.get(a) == v for a in holders):
            session.sql(f"ALTER MODEL {MODEL_FQN} VERSION {v} UNSET ALIAS").collect()
    for v, a in mapping.items():
        session.sql(f"ALTER MODEL {MODEL_FQN} VERSION {v} SET ALIAS = {a}").collect()

def versions_view():
    v = show(f"SHOW VERSIONS IN MODEL {MODEL_FQN}")
    v["metrics"] = v["metadata"].map(lambda m: {k: round(x, 4) for k, x in json.loads(m).get("metrics", {}).items()
                                                  if isinstance(x, (int, float))})
    return v[["name", "aliases", "is_default_version", "created_on", "metrics", "inference_services"]].set_index("name")

# Known starting state: V2 is production (it backs the REST service), V3 (Section 5) is the challenger
assign_aliases({SERVING_VERSION: "PRODUCTION", TRAIN_VERSION: "CHALLENGER"})
versions_view()

In [ ]:
probe_ids = test_pdf["LOAN_ID"].head(5).tolist()
probe_sql = f"""
SELECT LOAN_ID,
       ROUND(MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA({', '.join(f'{c}::FLOAT' for c in FEATURES)}):output_feature_1, 4) AS via_production_alias,
       ROUND(MODEL({MODEL_FQN}, V2)!PREDICT_PROBA({', '.join(f'{c}::FLOAT' for c in FEATURES)}):output_feature_1, 4) AS pinned_v2,
       ROUND(MODEL({MODEL_FQN}, V3)!PREDICT_PROBA({', '.join(f'{c}::FLOAT' for c in FEATURES)}):output_feature_1, 4) AS pinned_v3
FROM {FV_TABLE} WHERE LOAN_ID IN ({', '.join(map(str, probe_ids))}) ORDER BY LOAN_ID"""
before = q(probe_sql)
print("SQL, before promotion: PRODUCTION alias answers with V2")
display(before)

# Python: the registry resolves the alias exactly like a version name
mv_prod = reg.get_model(MODEL_NAME).version("PRODUCTION")
print("Python: reg.get_model(...).version('PRODUCTION') ->", mv_prod.version_name)
py_pred = mv_prod.run(test_pdf[test_pdf["LOAN_ID"].isin(probe_ids)][["LOAN_ID", *FEATURES]].sort_values("LOAN_ID"),
                      function_name="predict_proba")
print("Python predictions:", py_pred["output_feature_1"].astype("float64").round(4).tolist())

# Review step (7f #5): score the drifted week with both aliases and compare business impact
review = q(f"""
WITH w AS (SELECT * FROM {INFERENCE_LOG} WHERE PREDICTION_TS >= DATEADD('day', -{DRIFT_DAYS}, CURRENT_DATE())),
s AS (
  SELECT ACTUAL_APPROVED, LOAN_AMOUNT,
         IFF(MODEL({MODEL_FQN}, PRODUCTION)!PREDICT_PROBA({', '.join(FEATURES)}):output_feature_1::FLOAT >= 0.5, 1, 0) AS p_prod,
         IFF(MODEL({MODEL_FQN}, CHALLENGER)!PREDICT_PROBA({', '.join(FEATURES)}):output_feature_1::FLOAT >= 0.5, 1, 0) AS p_chal
  FROM w)
SELECT 'PRODUCTION (V2)' AS candidate, ROUND(AVG(IFF(p_prod = ACTUAL_APPROVED, 1, 0)), 4) AS accuracy,
       ROUND(SUM(IFF(p_prod = 1 AND ACTUAL_APPROVED = 0, LOAN_AMOUNT, 0)) / 1e6, 1) AS misapproved_musd
FROM s
UNION ALL
SELECT 'CHALLENGER (V3)', ROUND(AVG(IFF(p_chal = ACTUAL_APPROVED, 1, 0)), 4),
       ROUND(SUM(IFF(p_chal = 1 AND ACTUAL_APPROVED = 0, LOAN_AMOUNT, 0)) / 1e6, 1)
FROM s""")
print(f"Review on the last {DRIFT_DAYS} (drifted) days:")
review

Neither version was trained on the drifted population, so the real fix is the retrain in 7f, which logs a new version as `CHALLENGER`. The mechanics of promotion are identical whichever version wins the review, so the next cells promote **V3** to show the cut-over, then roll it back.

In [ ]:
assign_aliases({TRAIN_VERSION: "PRODUCTION", SERVING_VERSION: "PREVIOUS_PRODUCTION"})
session.sql(f"""UPDATE {REVIEW_QUEUE} SET STATUS = 'PROMOTED', RESOLUTION = 'PRODUCTION alias moved {SERVING_VERSION} -> {TRAIN_VERSION}'
                WHERE STATUS = 'OPEN'""").collect()

after = q(probe_sql)          # byte-for-byte the same SQL as before
print("SQL, after promotion: the same query now answers with V3")
display(after)
assert (after["VIA_PRODUCTION_ALIAS"] == after["PINNED_V3"]).all()
print("Python resolves PRODUCTION ->", reg.get_model(MODEL_NAME).version("PRODUCTION").version_name)
versions_view()

In [ ]:
assign_aliases({SERVING_VERSION: "PRODUCTION", TRAIN_VERSION: "CHALLENGER"})
session.sql(f"""UPDATE {REVIEW_QUEUE} SET STATUS = 'CLOSED', CLOSED_AT = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
                RESOLUTION = RESOLUTION || '; rolled back to {SERVING_VERSION} pending retrained challenger (demo)'
                WHERE STATUS = 'PROMOTED'""").collect()

rolled_back = q(probe_sql)
assert (rolled_back["VIA_PRODUCTION_ALIAS"] == rolled_back["PINNED_V2"]).all()
print("Rolled back: PRODUCTION ->", reg.get_model(MODEL_NAME).version("PRODUCTION").version_name)

# Who moved which alias, and when: alias changes are ordinary, audited DDL
display(q(f"""
SELECT start_time, role_name, REGEXP_REPLACE(query_text, '\\\\s+', ' ') AS statement
FROM TABLE({DB}.INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 1000))
WHERE query_text ILIKE 'ALTER MODEL%ALIAS%' AND execution_status = 'SUCCESS'
ORDER BY start_time DESC LIMIT 8"""))
display(q(f"SELECT TICKET_ID, STATUS, RESOLUTION, CLOSED_AT FROM {REVIEW_QUEUE} ORDER BY TICKET_ID DESC LIMIT 1"))
versions_view()

### 8b. Aliases with inference services and gateways

* **Warehouse inference** (SQL, dbt, Dynamic Tables, `mv.run`) resolves the alias **on every call**, so promotion takes effect immediately, as shown above.
* **An inference service is bound to one version.** `model.version("PRODUCTION").create_service(...)` resolves the alias **when the service is deployed**. Moving the alias later doesn't change the running container, which is what you want for a reproducible, debuggable endpoint.
* **For a stable REST URL across promotions,** put a **Snowflake Gateway** in front. Each version gets its own service, and the gateway's spec decides which one receives traffic: 100/0, a 90/10 canary, or shadow traffic. Promotion becomes `ALTER GATEWAY … FROM SPECIFICATION`, and clients keep calling the same hostname. Pair it with the alias: `PRODUCTION` points at the version whose service carries the primary traffic.
* **Monitors are per version.** After promoting, create a monitor for the new version, and suspend the old version's monitor once it has drained.

[Gateway Documentation](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/gateway-monitor-and-ab-testing)

In [ ]:
RUN_SERVICE_PROMOTION_EXAMPLE = False   # builds a V3 image + service (~10-20 min, SPCS credits) and a gateway

GATEWAY_NAME = f"{DB}.{SCHEMA}.MORTGAGE_APPROVAL_GATEWAY"
CHALLENGER_SERVICE = f"{DB}.{SCHEMA}.{SERVICE_NAME}_{TRAIN_VERSION}"

def gateway_spec(primary_svc, challenger_svc, challenger_pct):
    return f"""
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: {primary_svc}!inference
    weight: {100 - challenger_pct}
  - type: endpoint
    value: {challenger_svc}!inference
    weight: {challenger_pct}
"""

if RUN_SERVICE_PROMOTION_EXAMPLE:
    # 1. One service per version; the alias resolves at deploy time
    reg.get_model(MODEL_NAME).version("CHALLENGER").create_service(
        service_name=CHALLENGER_SERVICE, service_compute_pool=COMPUTE_POOL,
        ingress_enabled=True, min_instances=1, max_instances=2)
    # 2. Canary: 90% production, 10% challenger, behind one permanent hostname
    session.sql(f"CREATE OR REPLACE GATEWAY {GATEWAY_NAME} FROM SPECIFICATION $${gateway_spec(SERVICE_FQN, CHALLENGER_SERVICE, 10)}$$").collect()
    # 3. Promote: shift all traffic to the challenger and move the alias in the same change window
    session.sql(f"ALTER GATEWAY {GATEWAY_NAME} FROM SPECIFICATION $${gateway_spec(SERVICE_FQN, CHALLENGER_SERVICE, 100)}$$").collect()
    assign_aliases({TRAIN_VERSION: "PRODUCTION", SERVING_VERSION: "PREVIOUS_PRODUCTION"})
    display(show(f"DESCRIBE GATEWAY {GATEWAY_NAME}"))
else:
    print("Illustrative: set RUN_SERVICE_PROMOTION_EXAMPLE = True to deploy CHALLENGER as a service behind a gateway.")
    print("Canary spec (90/10):", gateway_spec(SERVICE_FQN, CHALLENGER_SERVICE, 10))

> **💡 Where Snowflake differentiates**
>
> * **Promotion is one governed statement.** `ALTER MODEL … SET ALIAS` moves production for every SQL and Python caller at once: no redeploy, no config push, no pipeline edits. Rollback is the same statement in reverse.
> * **The audit trail is automatic.** Alias moves are DDL in query history, attributed to a user and role, and lineage still pins each prediction table to an exact version.
> * **Batch and real-time follow the same contract.** Aliases drive warehouse inference directly, and a Gateway gives REST callers one stable URL while traffic shifts between versioned services.

## 9. Governance: one RBAC model from raw data to production model

> **🔐 Every object in this notebook is a Snowflake securable governed by the same role-based access control.** There's no second permission system for the ML stack, and no copies of data or models living outside the account.
>
> | Lifecycle step | Object(s) | How access is controlled |
> |---|---|---|
> | Raw and cleaned data (1, 1d) | `MORTGAGE_LENDING_DEMO_DATA`, `MORTGAGE_APPLICATIONS_CLEAN` | `SELECT` grants. Masking and row-access policies are enforced for every role that reads them, whether analyst, pipeline, or training job. |
> | Ingestion and prep (1d) | Procedures, task, network rule, secret, EAI | `USAGE` on procedures. The secret is usable only by roles granted `USAGE` on it (the procedure owner); egress is limited to hosts in the network rule; `OPERATE` on tasks. |
> | Features and DQ (2, 3) | Feature View (a Dynamic Table), data metric functions | `SELECT` on the feature view; DQ results are visible only to roles granted access. |
> | Training data (5) | Dataset `MORTGAGE_APPROVAL_TRAINING_DS` | Grants on the dataset object; versions are immutable. |
> | Experiments (5) | `MORTGAGE_APPROVAL_EXPERIMENT` | Grants on the experiment; runs and params are Snowflake objects, not a separate tracking server. |
> | Model (4, 5, 8) | `MORTGAGE_APPROVAL_CLASSIFIER` | `USAGE` / `READ` / `OWNERSHIP` (see the access model below). |
> | Serving (6) | Service, endpoints, gateway, compute pool | Service role `ALL_ENDPOINTS_USAGE` to call the endpoint; `USAGE` on the compute pool; `BIND SERVICE ENDPOINT` for public ingress. |
> | Monitoring and alerting (7) | Model monitor, alert, review queue | Any privilege on the monitor for `SHOW`/`DESCRIBE`; `MODIFY` to alter; `OWNERSHIP` to drop. The alert owner needs `EXECUTE ALERT` on the account and `USAGE` on the email integration. |
>
> Lineage (5d) and query and access history audit every hop, so one question ("who could see, train on, or score with applicant income?") is answered in SQL, not by reconciling four systems.

### The access model on the model object itself

These are **not one role**: running inference and seeing a model's internals and metrics are separate grants.

| Privilege on the model | Run inference | See versions (`SHOW MODELS`, `SHOW VERSIONS`) | See metadata: comments, tags, **metrics** | Download artifacts and files | SPCS inference and deploying services | Manage versions, aliases, default, metrics |
|---|:-:|:-:|:-:|:-:|:-:|:-:|
| **USAGE** | ✅ warehouse (SQL, `mv.run`) | ✅ | ❌ | ❌ | ❌ | ❌ |
| **READ** | ✅ warehouse + SPCS | ✅ | ✅ | ✅ | ✅ | ❌ |
| **OWNERSHIP** | ✅ | ✅ | ✅ | ✅ | ✅ | ✅ |

* **"Can score but can't inspect"** = `USAGE`: the right grant for an app or business role. The docs describe it as warehouse inference "without being able to see any of its internals".
* **"Can review the model and its accuracy"** = `READ`: the right grant for a validation or risk team. It covers metrics, comments, tags, and files.
* **"Can promote"** = `OWNERSHIP`, because only the owner can move aliases. If promotion must sit with a separate production role, use the documented **tag-based** pattern: production owns a `LIVE_VERSION` tag and callers resolve the version from it.
* **Expected inputs (the signature).** The docs say which privileges allow inference and which reveal metadata and files, but **not** whether a `USAGE`-only role can list a function's input signature (`SHOW FUNCTIONS IN MODEL`). Confirm with a test role before relying on it as a control.
* **Monitors and services are separate securables,** so "who can see the accuracy dashboard" (privileges on the monitor) and "who can call the REST endpoint" (the service role) are granted independently of the model.

```sql
-- Illustrative role design (not executed here)
GRANT USAGE ON MODEL ML_DEMO.PUBLIC.MORTGAGE_APPROVAL_CLASSIFIER TO ROLE LOAN_APP_SCORER;    -- score only
GRANT USAGE ON WAREHOUSE COMPUTE_WH                                TO ROLE LOAN_APP_SCORER;
GRANT SERVICE ROLE ML_DEMO.PUBLIC.MORTGAGE_APPROVAL_SERVICE!ALL_ENDPOINTS_USAGE TO ROLE LOAN_APP_SCORER;  -- REST

GRANT READ ON MODEL ML_DEMO.PUBLIC.MORTGAGE_APPROVAL_CLASSIFIER TO ROLE MODEL_RISK_REVIEWER;  -- metrics, metadata, files
GRANT SELECT ON TABLE ML_DEMO.PUBLIC.MORTGAGE_V2_INFERENCE_LOG  TO ROLE MODEL_RISK_REVIEWER;  -- review evidence

GRANT USAGE ON FUTURE MODELS IN SCHEMA ML_DEMO.PUBLIC TO ROLE LOAN_APP_SCORER;               -- new models inherit
-- Promotion stays with the model owner role (ML_PLATFORM_OWNER), which alone can run ALTER MODEL ... SET ALIAS
```

In [ ]:
def grants_on(kind, name):
    try:
        g = show(f"SHOW GRANTS ON {kind} {name}")
        return g.assign(object=f"{kind} {name.split('.')[-1]}")[["object", "privilege", "granted_to", "grantee_name"]]
    except Exception as e:
        return pd.DataFrame([{"object": f"{kind} {name.split('.')[-1]}", "privilege": "n/a",
                              "granted_to": "", "grantee_name": str(e).splitlines()[0][:80]}])

pd.concat([grants_on("MODEL", MODEL_FQN),
           grants_on("MODEL MONITOR", MONITOR_NAME),
           grants_on("TABLE", SOURCE_TABLE),
           grants_on("TABLE", INFERENCE_LOG)], ignore_index=True)

## 10. Summary and cleanup

| Requirement | Where it's shown | Snowflake capability |
|---|---|---|
| Explore with pandas and SQL, visible in query history | 1a–1c, query-history cell | Snowpark pushdown, `QUERY_HISTORY` |
| Bring data in and prepare it with stored procedures | 1d | Python procedure + EAI + secret; SQL procedure with `MERGE`; tasks |
| SQL behind the Feature Store features | 2 | Feature View → Dynamic Table |
| Data-quality monitoring on the feature store | 3 | Data metric functions, expectations, `DATA_METRIC_SCAN` |
| Import a model trained elsewhere and reproduce its baseline | 4 | Registry `log_model` for native types, parity test |
| Dataset reproducibility, experiment rerun within tolerance | 5a–5c | Immutable Datasets, Experiment Tracking |
| Data → Feature Store → Dataset → Model lineage | 5d | `GET_LINEAGE`, Snowsight lineage graph |
| Command-line calls and latency | 6a–6b | REST endpoint + PAT, service functions, warehouse inference |
| Scaling across instances and load balancing | 6b-ii | `MIN/MAX_INSTANCES`, autoscaling, managed ingress |
| Failure modes | 6c | Bad payloads, instance loss, cold start, auth, bad SQL rows |
| Operational and model metrics, plus custom metrics | 7a–7d | Model monitor (`CUSTOM_METRIC_COLUMNS`, segments), service metrics |
| Simulated drift, threshold breach, owner alert | 7a, 7e | Drift metrics, alert, email integration, review queue |
| Retraining and review workflow | 7f | Runbook, retrain procedure, task graph |
| Promote and demote with aliases | 8 | `ALTER MODEL … SET ALIAS`, `MODEL(m, PRODUCTION)`, Gateway |
| Governance across the lifecycle | 9 | One RBAC model: USAGE / READ / OWNERSHIP on models |

**Cleanup.** The cell below returns the demo to a low-cost resting state: service at 1 instance and suspended, compute pool, monitor, alert, and tasks suspended. Set `DROP_DEMO_OBJECTS = True` to also remove the objects created in Sections 1d, 6, and 7.

In [ ]:
RUN_CLEANUP       = True    # suspend compute; safe to rerun
DROP_DEMO_OBJECTS = False   # also drop the objects this notebook created in 1d, 6 and 7

def try_sql(stmt):
    try:
        session.sql(stmt).collect()
        return "ok"
    except Exception as e:
        return str(e).splitlines()[0][:90]

actions = []
if RUN_CLEANUP:
    if service_exists():
        actions += [(s, try_sql(s)) for s in [f"ALTER SERVICE {SERVICE_FQN} SET MIN_INSTANCES = 1 MAX_INSTANCES = 1",
                                              f"ALTER SERVICE {SERVICE_FQN} SUSPEND"]]
    actions += [(s, try_sql(s)) for s in [
        f"ALTER MODEL MONITOR {MONITOR_NAME} SUSPEND",
        f"ALTER ALERT {ALERT_NAME} SUSPEND",
        f"ALTER TASK IF EXISTS {DB}.{SCHEMA}.MORTGAGE_PREP_TASK SUSPEND",
        f"ALTER TASK IF EXISTS {DB}.{SCHEMA}.MORTGAGE_RETRAIN_TASK SUSPEND",
    ]]
    # The pool is shared: only suspend it when nothing else on it is still running
    others = show(f"SHOW SERVICES IN COMPUTE POOL {COMPUTE_POOL}")
    running = [] if others.empty else others[~others["status"].isin(["SUSPENDED", "DONE", "FAILED", "DELETED"])]["name"].tolist()
    actions.append((f"ALTER COMPUTE POOL {COMPUTE_POOL} SUSPEND",
                    try_sql(f"ALTER COMPUTE POOL {COMPUTE_POOL} SUSPEND") if not running
                    else f"skipped: still running on the pool -> {running}"))
if DROP_DEMO_OBJECTS:
    actions += [(s, try_sql(s)) for s in [
        f"DROP ALERT IF EXISTS {ALERT_NAME}",
        f"DROP MODEL MONITOR IF EXISTS {MONITOR_NAME}",
        f"DROP TABLE IF EXISTS {INFERENCE_LOG}", f"DROP TABLE IF EXISTS {BASELINE_TBL}",
        f"DROP TABLE IF EXISTS {REVIEW_QUEUE}", f"DROP TABLE IF EXISTS {CLEAN_TABLE}",
        f"DROP TASK IF EXISTS {DB}.{SCHEMA}.MORTGAGE_EVALUATE_CHALLENGER_TASK",
        f"DROP TASK IF EXISTS {DB}.{SCHEMA}.MORTGAGE_RETRAIN_TASK",
        f"DROP TASK IF EXISTS {DB}.{SCHEMA}.MORTGAGE_INGEST_TASK",
        f"DROP TASK IF EXISTS {DB}.{SCHEMA}.MORTGAGE_PREP_TASK",
        f"DROP PROCEDURE IF EXISTS {DB}.{SCHEMA}.NOTIFY_MODEL_OWNER()",
        f"DROP PROCEDURE IF EXISTS {DB}.{SCHEMA}.PREP_MORTGAGE_APPLICATIONS(VARCHAR, VARCHAR)",
        f"DROP STAGE IF EXISTS {DB}.{SCHEMA}.MORTGAGE_CLI",
    ]]

pd.DataFrame(actions, columns=["statement", "result"])